In [3]:
import cdsapi
import xarray as xr
import pandas as pd
import os
import time
import numpy as np

# --- 1. SABİT PARAMETRELER ---
YEARS = ['2020', '2021', '2022'] # İstediğiniz 36 ay için 2020, 2021, 2022
FILE_DIR = 'era5_uv_files' # İndirilen NetCDF dosyalarının kaydedileceği klasör
EXCEL_FILE_NAME = 'ERA5_Aylik_UV_Index_Raporu.xlsx'

# ERA5 değişken adı - UV index hesaplamak için solar radiation kullanacağız
ERA5_VARIABLE = "downward_uv_radiation_at_the_surface" 
SSRD_SHORT_NAME = 'uvb'

# --- 2. Şehir Alanları ---
ERA5_CITY_AREAS = {
    # 🌍 ABD Eyalet Başkentleri ve Bölgeler (1-55)
    'Birmingham - Alabama': '33.7700/-87.0600/33.2700/-86.5600',
    'Anchorage - Alaska': '61.4200/-150.1700/60.9200/-149.6700'
}

def solar_to_uv_index(solar_radiation):
    """
    Solar radiation (W/m²) değerini UV Index'e dönüştürür
    Yaklaşık formül: UV Index ≈ Solar Radiation * 0.025
    Bu formül, tipik solar radiation değerlerini (0-800 W/m²) UV Index (0-11) aralığına dönüştürür
    """
    uv_index = solar_radiation * 0.025
    # UV Index 0-11 arasında olmalı, aşırı değerleri sınırla
    uv_index = np.clip(uv_index, 0, 15)  # 15 üst sınır, nadiren 11'i aşabilir
    return np.round(uv_index, 2)

# --- Kısım 3: Toplu Veri İndirme Fonksiyonu ---
def era5_bulk_download(city_name, area_box):
    """Tek bir şehir için ERA5 verisini indirir ve NetCDF dosya yolunu döndürür."""
    
    file_name = os.path.join(FILE_DIR, f'{city_name.replace(" ", "_").replace("/", "_")}_uv.nc')
    
    # Dosya zaten varsa indirme adımını atla
    if os.path.exists(file_name):
        print(f"⏩ {city_name} için dosya zaten var. Atlanıyor.")
        return file_name
        
    c = cdsapi.Client()
    print(f"🌍 {city_name} (Area: {area_box}) için solar radiation indirme başlatılıyor...")

    try:
        c.retrieve(
            'reanalysis-era5-single-levels-monthly-means', 
            {
                'product_type': 'monthly_averaged_reanalysis',
                'variable': ERA5_VARIABLE,
                'year': YEARS,
                'month': [f'{i:02d}' for i in range(1, 13)],
                'time': '12:00',  # UV için gün ortası daha iyi
                'data_format': 'netcdf',
                'area': area_box, 
            },
            file_name)
        
        print(f"✅ {city_name} solar radiation indirme başarılı.")
        return file_name
        
    except Exception as e:
        print(f"❌ {city_name} indirme hatası: {e}")
        return None

# --- Kısım 4: NetCDF Dosyasını İşleme Fonksiyonu ---
def process_netcdf_to_series(file_path, city_name):
    """İndirilen NetCDF dosyasını Pandas Serisine dönüştürür ve UV Index'e çevirir."""
    if file_path is None or not os.path.exists(file_path):
        return None
        
    try:
        ds = xr.open_dataset(file_path)
        
        print(f"📊 {city_name} için mevcut değişkenler: {list(ds.variables.keys())}")
        
        # Solar radiation değişkenini bul
        solar_variable = None
        for var_name in ds.variables.keys():
            if 'uvb' in var_name.lower() or var_name == SSRD_SHORT_NAME:
                solar_variable = var_name
                break
        
        if solar_variable is None:
            # Eğer ssrd bulunamazsa, ilk data değişkenini kullan
            for var_name in ds.variables.keys():
                if var_name not in ['time', 'latitude', 'longitude']:
                    solar_variable = var_name
                    break
        
        if solar_variable is None:
            print(f"⚠️ {city_name} için solar radiation değişkeni bulunamadı.")
            return None
        
        print(f"📈 {city_name} için kullanılan değişken: {solar_variable}")
        
        # Solar radiation değerlerini al (W/m²)
        solar_data = ds[solar_variable]
        
        # Ham solar radiation değerlerini kontrol et
        solar_mean = float(solar_data.mean(['latitude', 'longitude']).mean('time').values)
        print(f"🔆 {city_name} ortalama solar radiation: {solar_mean:.2f} W/m²")
        
        # Alan (0.5x0.5 derece) üzerindeki ortalamayı hesapla
        monthly_mean_solar = solar_data.mean(['latitude', 'longitude'])
        
        # Solar radiation'ı UV Index'e dönüştür
        monthly_uv_index = solar_to_uv_index(monthly_mean_solar)
        
        # Pandas Serisine dönüştür (2 ondalık basamağa yuvarla)
        monthly_data = monthly_uv_index.to_series().round(2)
        
        # Seriyi yeniden adlandır (Excel'de sütun adı için kullanılacak)
        monthly_data.name = city_name 
        
        print(f"📊 {city_name} UV Index aralığı: {monthly_data.min():.2f} - {monthly_data.max():.2f}")
        
        return monthly_data
        
    except Exception as e:
        print(f"⚠️ {city_name} verisi işlenirken hata oluştu: {e}")
        return None

# --- Kısım 5: UV Index Analizi ---
def analyze_uv_index(df):
    """UV Index değerlerini analiz eder ve istatistikleri yazdırır."""
    print(f"\n📊 UV INDEX ANALİZİ")
    print(f"═" * 50)
    
    # Temel istatistikler
    print(f"📈 Genel İstatistikler:")
    print(f"   Minimum UV Index: {df.min().min():.2f}")
    print(f"   Maksimum UV Index: {df.max().max():.2f}")
    print(f"   Ortalama UV Index: {df.mean().mean():.2f}")
    print(f"   Medyan UV Index: {df.median().median():.2f}")
    
    # UV Index kategorileri
    uv_categories = {
        'Düşük (0-2)': (0, 2),
        'Orta (3-5)': (3, 5),
        'Yüksek (6-7)': (6, 7),
        'Çok Yüksek (8-10)': (8, 10),
        'Aşırı (11+)': (11, 20)
    }
    
    print(f"\n📋 UV Index Dağılımı:")
    total_points = df.size
    for category, (low, high) in uv_categories.items():
        if high == 20:  # Aşırı kategori için
            count = (df >= low).sum().sum()
        else:
            count = ((df >= low) & (df <= high)).sum().sum()
        
        percentage = (count / total_points) * 100
        print(f"   {category}: {count:3d} veri noktası (%{percentage:5.1f})")
    
    # Mevsimsel analiz
    print(f"\n🌤️  Mevsimsel Ortalamalar:")
    seasons = {
        'Kış (Aralık-Şubat)': ['12', '01', '02'],
        'İlkbahar (Mart-Mayıs)': ['03', '04', '05'],
        'Yaz (Haziran-Ağustos)': ['06', '07', '08'],
        'Sonbahar (Eylül-Kasım)': ['09', '10', '11']
    }
    
    for season, months in seasons.items():
        season_columns = []
        for year in YEARS:
            for month in months:
                col_name = f'{year}-{month}'
                if col_name in df.columns:
                    season_columns.append(col_name)
        
        if season_columns:
            season_avg = df[season_columns].mean().mean()
            print(f"   {season}: {season_avg:.2f} UV Index")

# --- Kısım 6: Ana Çalışma Bölümü ---
if __name__ == "__main__":
    # Çıktı klasörünü oluştur
    os.makedirs(FILE_DIR, exist_ok=True)
    
    # Tüm şehirlerin verilerini tutacak liste
    all_city_data = []

    print(f"*** Toplam {len(ERA5_CITY_AREAS)} şehir için UV Index veri indirme ve işleme başlatılıyor. ***")
    print(f"🔬 Kullanılan yöntem: Solar Radiation → UV Index dönüşümü")
    
    # Şehirler üzerinde döngü
    for city_name, area_box in ERA5_CITY_AREAS.items():
        
        # 1. Veriyi İndir
        netcdf_path = era5_bulk_download(city_name, area_box)
        
        if netcdf_path:
            # 2. Veriyi İşle
            city_series = process_netcdf_to_series(netcdf_path, city_name)
            
            if city_series is not None:
                all_city_data.append(city_series)
        
        # API yorgunluğunu önlemek için kısa bir bekleme ekle
        time.sleep(1) 

    print("\n*** Tüm şehirlerin verileri toplandı. Excel dosyası oluşturuluyor... ***")

    if not all_city_data:
        print("⛔ Hiçbir şehir için veri toplanamadı. Lütfen API anahtarınızı ve internet bağlantınızı kontrol edin.")
    else:
        # Pandas Series listesini tek bir DataFrame'e birleştirme
        combined_df = pd.DataFrame(all_city_data)
        final_df = combined_df
        
        # Sütun isimlerini 1. ay, 2. ay, ... 36. ay olarak ayarlayın
        month_names = [f'{y}-{m:02d}' for y in YEARS for m in range(1, 13)]
        final_df.columns = month_names[:len(final_df.columns)]
        
        # Index'i Şehir adı olarak tutmak için index'i yeniden adlandırın
        final_df.index.name = 'City'
        
        # UV Index analizi yap
        analyze_uv_index(final_df)
        
        # Excel'e kaydet
        try:
            final_df.to_excel(EXCEL_FILE_NAME)
            print(f"\n🎉 BAŞARILI: '{EXCEL_FILE_NAME}' dosyası oluşturuldu.")
            print(f"Toplam {len(final_df)} şehir ({len(final_df.columns)} aylık UV Index verisi) raporlandı.")
            
            # Örnek verileri göster
            print(f"\n📋 İlk 6 ay için örnek UV Index değerleri:")
            print(final_df.iloc[:, :6])
            
        except Exception as e:
            print(f"❌ Excel'e yazma hatası: {e}")

*** Toplam 2 şehir için UV Index veri indirme ve işleme başlatılıyor. ***
🔬 Kullanılan yöntem: Solar Radiation → UV Index dönüşümü
🌍 Birmingham - Alabama (Area: 33.7700/-87.0600/32.2700/-85.5600) için solar radiation indirme başlatılıyor...


2025-11-26 23:28:55,646 INFO Request ID is a1335558-291a-46c1-825d-1890152bafbf
2025-11-26 23:28:55,752 INFO status has been updated to accepted
2025-11-26 23:29:09,547 INFO status has been updated to failed


❌ Birmingham - Alabama indirme hatası: 400 Client Error: Bad Request for url: https://cds.climate.copernicus.eu/api/retrieve/v1/jobs/a1335558-291a-46c1-825d-1890152bafbf/results
The job has failed
MARS returned no data, please check your selection.Request submitted to the MARS server:
[{'area': '33.7700/-87.0600/32.2700/-85.5600', 'dataset': 'reanalysis-monthly-means-of-daily-means', 'time': '12:00:00', 'param': '57', 'class': ['ea'], 'expect': ['any'], 'number': ['all'], 'levtype': ['sfc'], 'date': ['2020-01-01', '2020-02-01', '2020-03-01', '2020-04-01', '2020-05-01', '2020-06-01', '2020-07-01', '2020-08-01', '2020-09-01', '2020-10-01', '2020-11-01', '2020-12-01', '2021-01-01', '2021-02-01', '2021-03-01', '2021-04-01', '2021-05-01', '2021-06-01', '2021-07-01', '2021-08-01', '2021-09-01', '2021-10-01', '2021-11-01', '2021-12-01', '2022-01-01', '2022-02-01', '2022-03-01', '2022-04-01', '2022-05-01', '2022-06-01', '2022-07-01', '2022-08-01', '2022-09-01', '2022-10-01', '2022-11-01', '202

2025-11-26 23:29:11,419 INFO Request ID is 3630cae5-9075-493b-aaa8-65e7afb7f6af
2025-11-26 23:29:11,519 INFO status has been updated to accepted
2025-11-26 23:29:25,208 INFO status has been updated to failed


❌ Anchorage - Alaska indirme hatası: 400 Client Error: Bad Request for url: https://cds.climate.copernicus.eu/api/retrieve/v1/jobs/3630cae5-9075-493b-aaa8-65e7afb7f6af/results
The job has failed
MARS returned no data, please check your selection.Request submitted to the MARS server:
[{'area': '61.4200/-150.1700/59.9200/-148.6700', 'dataset': 'reanalysis-monthly-means-of-daily-means', 'time': '12:00:00', 'param': '57', 'class': ['ea'], 'expect': ['any'], 'number': ['all'], 'levtype': ['sfc'], 'date': ['2020-01-01', '2020-02-01', '2020-03-01', '2020-04-01', '2020-05-01', '2020-06-01', '2020-07-01', '2020-08-01', '2020-09-01', '2020-10-01', '2020-11-01', '2020-12-01', '2021-01-01', '2021-02-01', '2021-03-01', '2021-04-01', '2021-05-01', '2021-06-01', '2021-07-01', '2021-08-01', '2021-09-01', '2021-10-01', '2021-11-01', '2021-12-01', '2022-01-01', '2022-02-01', '2022-03-01', '2022-04-01', '2022-05-01', '2022-06-01', '2022-07-01', '2022-08-01', '2022-09-01', '2022-10-01', '2022-11-01', '202

In [12]:
import xarray as xr
import numpy as np
import cdsapi
import os

# --- 1. CDS API Client ---
c = cdsapi.Client()

# --- 2. Dataset ve İndirme Talebi ---
dataset = "reanalysis-era5-single-levels"  # Günlük/saatlik ERA5
request = {
    "product_type": "reanalysis",
    "variable": "downward_uv_radiation_at_the_surface",
    "year": "2021",
    "month": "08",
    "day": "01",
    "time": "13:00",
    "area": [41.00,28.00,39.00,30.00],  # [N, W, S, E]
    "format": "netcdf"  # Dosya formatı
}

# --- 3. Dosya adı ---
output_file = "era5_uv_20210701_13.nc"

# --- 4. Veri indirme ---
print(f"🌍 Veri indiriliyor: {output_file} ...")
c.retrieve(dataset, request, output_file)
print("✅ İndirme tamamlandı.")

# --- 5. Dosya uzantısını göster ---
file_ext = os.path.splitext(output_file)[1]
print(f"📁 Dosya uzantısı: {file_ext}")



file_path = "era5_uv_20210701_13.nc"
ds = xr.open_dataset(file_path)
uvb = ds['uvb']  # J/m²

# --- 1. Alan ortalaması ---
uvb_mean_area = uvb.mean(['latitude', 'longitude']).values.item()  # J/m²

# --- 2. Saatlik UV gücü (W/m²) ---
uvb_w_per_m2 = uvb_mean_area / 3600  # 1 saat = 3600 s

# --- 3. UV Index hesapla ---
uv_index = uvb_w_per_m2 * 0.025
uv_index = np.clip(uv_index, 0, 15)
uv_index = round(uv_index, 2)

print(f"🌍 Ortalama UVB (J/m²): {uvb_mean_area:.2f}")
print(f"🔹 Ortalama UV Index: {uv_index}")


🌍 Veri indiriliyor: era5_uv_20210701_13.nc ...


2025-11-26 23:51:43,424 INFO Request ID is 2a3b92b9-4fca-416f-9d48-2883cca76a86
2025-11-26 23:51:43,538 INFO status has been updated to accepted
2025-11-26 23:51:57,307 INFO status has been updated to successful


4c9e09f1dc2e160974c79eb8bb71114f.nc:   0%|          | 0.00/24.8k [00:00<?, ?B/s]

✅ İndirme tamamlandı.
📁 Dosya uzantısı: .nc
🌍 Ortalama UVB (J/m²): 313491.47
🔹 Ortalama UV Index: 2.18


In [8]:
import cdsapi
import os

# --- 1. CDS API Client ---
c = cdsapi.Client()

# --- 2. Dataset ve İndirme Talebi ---
dataset = "reanalysis-era5-single-levels"  # Günlük/saatlik ERA5
request = {
    "product_type": "reanalysis",
    "variable": "downward_uv_radiation_at_the_surface",
    "year": "2021",
    "month": "01",
    "day": "01",
    "time": "13:00",
    "area": [41.00,28.00,39.00,30.00],  # [N, W, S, E]
    "format": "netcdf"  # Dosya formatı
}

# --- 3. Dosya adı ---
output_file = "era5_uv_20210701_13.nc"

# --- 4. Veri indirme ---
print(f"🌍 Veri indiriliyor: {output_file} ...")
c.retrieve(dataset, request, output_file)
print("✅ İndirme tamamlandı.")

# --- 5. Dosya uzantısını göster ---
file_ext = os.path.splitext(output_file)[1]
print(f"📁 Dosya uzantısı: {file_ext}")


🌍 Veri indiriliyor: era5_uv_20210701_13.nc ...


2025-11-26 23:47:53,855 INFO Request ID is 4d3b0686-e19e-4fb6-bf45-72d24ece5217
2025-11-26 23:47:54,092 INFO status has been updated to accepted
2025-11-26 23:48:07,770 INFO status has been updated to running
2025-11-26 23:48:15,458 INFO status has been updated to successful


f5ae3cd1522c0e126758a02c3742f1c.nc:   0%|          | 0.00/24.8k [00:00<?, ?B/s]

✅ İndirme tamamlandı.
📁 Dosya uzantısı: .nc


In [15]:
import cdsapi
import xarray as xr
import numpy as np
import os

# --- 1. CDS API Client ---
c = cdsapi.Client()

# --- 2. Şehir alanı ve dosya adı ---
area = [41.00, 28.00, 39.00, 30.00]  # [N, W, S, E]
file_name = "era5_uv_2021_monthly.nc"

# --- 3. Dosya yoksa indir ---
if not os.path.exists(file_name):
    print(f"🌍 Veriler indiriliyor: {file_name} ...")
    c.retrieve(
        'reanalysis-era5-single-levels-monthly-means',
        {
            'product_type': 'monthly_averaged_reanalysis',
            'variable': 'downward_uv_radiation_at_the_surface',
            'year': ['2021'],
            'month': [f"{m:02d}" for m in range(1, 13)],
            'time': '00:00',
            'format': 'netcdf',
            'area': area
        },
        file_name
    )
    print("✅ İndirme tamamlandı.")
else:
    print("⏩ Dosya zaten var, indirme atlandı.")

# --- 4. Dosyayı aç ---
ds = xr.open_dataset(file_name)

# --- 5. UVB değişkeni ---
uvb = ds['uvb']  # J/m² (aylık ortalama)

# --- 6. Aylık UV Index hesapla ---
uv_index_list = []
month_names = [f"{m:02d}" for m in range(1, 13)]

for i, month in enumerate(month_names):
    # Her ayın grid ortalaması
    uvb_mean = uvb.isel(valid_time=i).mean(['latitude', 'longitude']).values.item()
    
    # monthly-means dataset’inde her ay zaten ortalama → sadece 0.025 ile çarpabiliriz
    uv_index = uvb_mean * 0.025
    uv_index = np.clip(uv_index, 0, 15)
    uv_index = round(uv_index, 2)
    uv_index_list.append(uv_index)

# --- 7. Sonuçları ekrana yazdır ---
print("\n📊 2021 Aylık Ortalama UV Index (Seçilen Alan):")
for month, uv in zip(month_names, uv_index_list):
    print(f"   {month}: {uv}")


⏩ Dosya zaten var, indirme atlandı.

📊 2021 Aylık Ortalama UV Index (Seçilen Alan):
   01: 15.0
   02: 15.0
   03: 15.0
   04: 15.0
   05: 15.0
   06: 15.0
   07: 15.0
   08: 15.0
   09: 15.0
   10: 15.0
   11: 15.0
   12: 15.0


In [18]:
import xarray as xr
import numpy as np
import os
import cdsapi

# --- CDS API ---
c = cdsapi.Client()
area = [41.00, 28.00, 39.00, 30.00]
file_name = "era5_uv_2021_monthly.nc"

# --- Dosya yoksa indir ---
if not os.path.exists(file_name):
    print(f"🌍 Veriler indiriliyor: {file_name} ...")
    c.retrieve(
        'reanalysis-era5-single-levels-monthly-means',
        {
            'product_type': 'monthly_averaged_reanalysis',
            'variable': 'downward_uv_radiation_at_the_surface',
            'year': ['2021'],
            'month': [f"{m:02d}" for m in range(1, 13)],
            'time': '12:00',
            'format': 'netcdf',
            'area': area
        },
        file_name
    )
    print("✅ İndirme tamamlandı.")
else:
    print("⏩ Dosya zaten var, indirme atlandı.")

# --- Dosyayı aç ---
ds = xr.open_dataset(file_name)
uvb = ds['uvb']  # J/m² (aylık toplam)

# --- Aylık UV Index hesaplama ---
uv_index_list = []
month_days = [31,28,31,30,31,30,31,31,30,31,30,31]  # 2021 için gün sayıları

for i, days in enumerate(month_days):
    uvb_mean = uvb.isel(valid_time=i).mean(['latitude','longitude']).values.item()
    
    seconds_in_month = days * 24 * 3600
    uv_power = uvb_mean / seconds_in_month  # W/m²
    
    uv_index = uv_power * 0.025
    uv_index = np.clip(uv_index, 0, 15)
    uv_index_list.append(round(uv_index,2))

# --- Sonuçları yazdır ---
month_names = [f"{m:02d}" for m in range(1, 13)]
print("\n📊 2021 Aylık Ortalama UV Index (Seçilen Alan):")
for month, uv in zip(month_names, uv_index_list):
    print(f"   {month}: {uv}")


🌍 Veriler indiriliyor: era5_uv_2021_monthly.nc ...


2025-11-27 00:02:25,161 INFO Request ID is 57b080c2-b84c-4c50-9e49-173cdcd2c450
2025-11-27 00:02:25,413 INFO status has been updated to accepted
2025-11-27 00:02:39,194 INFO status has been updated to failed


HTTPError: 400 Client Error: Bad Request for url: https://cds.climate.copernicus.eu/api/retrieve/v1/jobs/57b080c2-b84c-4c50-9e49-173cdcd2c450/results
The job has failed
The 'format' key for requests is deprecated, please use 'data_format' instead. Use of 'format' is no longer part of the system testing, therefore it is not guaranteed to work.
MARS returned no data, please check your selection.Request submitted to the MARS server:
[{'area': [41.0, 28.0, 39.0, 30.0], 'dataset': 'reanalysis-monthly-means-of-daily-means', 'time': '12:00:00', 'param': '57', 'class': ['ea'], 'expect': ['any'], 'number': ['all'], 'levtype': ['sfc'], 'date': ['2021-01-01', '2021-02-01', '2021-03-01', '2021-04-01', '2021-05-01', '2021-06-01', '2021-07-01', '2021-08-01', '2021-09-01', '2021-10-01', '2021-11-01', '2021-12-01']}]

The job failed with: MarsNoDataError

In [10]:
import xarray as xr
import numpy as np
import os
import cdsapi

# --- CDS API ---
c = cdsapi.Client()
area = [41.00, 28.00, 39.00, 30.00]
file_name = "era5_uv_2021_monthly.nc"

# --- Dosya yoksa indir ---
if not os.path.exists(file_name):
    print(f"🌍 Veriler indiriliyor: {file_name} ...")
    c.retrieve(
        'reanalysis-era5-single-levels-monthly-means',
        {
            'product_type': 'monthly_averaged_reanalysis',
            'variable': 'downward_uv_radiation_at_the_surface',
            'year': ['2021'],
            'month': [f"{m:02d}" for m in range(1, 13)],
            'time': '00:00',
            'format': 'netcdf',
            'area': area
        },
        file_name
    )
    print("✅ İndirme tamamlandı.")
else:
    print("⏩ Dosya zaten var, indirme atlandı.")

# --- Dosyayı aç ---
ds = xr.open_dataset(file_name)
uvb = ds['uvb']  # J/m² (aylık toplam)

# --- Aylık UV Index hesaplama ---
month_days = [31,28,31,30,31,30,31,31,30,31,30,31]  # 2021 için gün sayıları
month_names = [f"{m:02d}" for m in range(1, 13)]
uv_index_list = []

print("\n🔍 HESAPLAMA ADIMLARI (Ay Ay):\n")

for i, (month, days) in enumerate(zip(month_names, month_days)):

    # 1. Aylık toplam UV (J/m²)
    uvb_mean = uvb.isel(valid_time=i).mean(['latitude','longitude']).values.item()

    # 2. Ay içindeki toplam saniye
    seconds_in_month = days * 24 * 3600

    # 3. Ortalama güç (W/m²)
    uv_power = uvb_mean / seconds_in_month

    # 4. UV Index
    uv_index = uv_power * 0.025
    uv_index = np.clip(uv_index, 0, 15)
    uv_index_list.append(round(uv_index, 2))

    # --- Her ay için hesaplama çıktısı ---
    print(f"📅 Ay: {month}")
    print(f"   • Aylık toplam UV (J/m²):     {uvb_mean:.2f}")
    print(f"   • Ay içindeki saniye sayısı:  {seconds_in_month}")
    print(f"   • Ortalama UV gücü (W/m²):    {uv_power:.6f}")
    print(f"   • UV Index:                    {uv_index:.2f}\n")

# --- Sonuçları yazdır ---
print("\n📊 2021 Aylık Ortalama UV Index (Seçilen Alan):")
for month, uv in zip(month_names, uv_index_list):
    print(f"   {month}: {uv}")


⏩ Dosya zaten var, indirme atlandı.

🔍 HESAPLAMA ADIMLARI (Ay Ay):

📅 Ay: 01
   • Aylık toplam UV (J/m²):     767021.06
   • Ay içindeki saniye sayısı:  2678400
   • Ortalama UV gücü (W/m²):    0.286373
   • UV Index:                    0.01

📅 Ay: 02
   • Aylık toplam UV (J/m²):     1222920.75
   • Ay içindeki saniye sayısı:  2419200
   • Ortalama UV gücü (W/m²):    0.505506
   • UV Index:                    0.01

📅 Ay: 03
   • Aylık toplam UV (J/m²):     1515842.88
   • Ay içindeki saniye sayısı:  2678400
   • Ortalama UV gücü (W/m²):    0.565951
   • UV Index:                    0.01

📅 Ay: 04
   • Aylık toplam UV (J/m²):     2031111.88
   • Ay içindeki saniye sayısı:  2592000
   • Ortalama UV gücü (W/m²):    0.783608
   • UV Index:                    0.02

📅 Ay: 05
   • Aylık toplam UV (J/m²):     2654701.00
   • Ay içindeki saniye sayısı:  2678400
   • Ortalama UV gücü (W/m²):    0.991152
   • UV Index:                    0.02

📅 Ay: 06
   • Aylık toplam UV (J/m²):     2706584.50


In [7]:
import xarray as xr
import numpy as np
import os
import cdsapi

# --- CDS API ---
c = cdsapi.Client()
area = [41.00, 28.00, 39.00, 30.00]   # bursa bölgesi
file_name = "era5_uv_2021_monthly.nc"

# --- Dosya yoksa indir ---
if not os.path.exists(file_name):
    print(f"🌍 Veriler indiriliyor: {file_name} ...")
    c.retrieve(
        'reanalysis-era5-single-levels-monthly-means',
        {
            'product_type': 'monthly_averaged_reanalysis',
            'variable': 'downward_uv_radiation_at_the_surface',
            'year': ['2021'],
            'month': [f"{m:02d}" for m in range(1, 13)],
            'time': '00:00',
            'format': 'netcdf',
            'area': area
        },
        file_name
    )
    print("✅ İndirme tamamlandı.")
else:
    print("⏩ Dosya zaten var, indirme atlandı.")

# --- Dosyayı aç ---
ds = xr.open_dataset(file_name)
uv = ds['uvb']  # ERA5 değişken adı = uvb  (J/m² / day)

# --- ERA5 açıklamasına göre:
#    Monthly means = günlük akümülasyon (= 24 saatlik toplam)
SECONDS_PER_DAY = 24 * 3600

month_names = [f"{m:02d}" for m in range(1, 13)]
uv_index_list = []

print("\n🔍 HESAPLAMA ADIMLARI (Ay Ay):\n")

print("\n🔍 HESAPLAMA ADIMLARI (Ay Ay):\n")

print("\n🔍 HESAPLAMA ADIMLARI (Ay Ay):\n")

for i, month in enumerate(month_names):

    # 1. Aylık UV (ERA5 monthly) → günlük toplam UV (J/m²/day)
    uv_j = uv.isel(valid_time=i).mean(['latitude','longitude']).values.item()

    # 2. W/m²'ye çevir (Her gün 86400 saniye)
    uv_wm2 = uv_j / SECONDS_PER_DAY

    # 3. Yaklaşık UV Index dönüşümü
    seasonal_k = [0.3, 0.3, 0.33, 0.33, 0.33, 0.35, 0.35, 0.35, 0.33, 0.33, 0.33, 0.3]
    k=seasonal_k[i]
    uv_index = uv_wm2 * k
    uv_index = np.clip(uv_index, 0, 11)

    uv_index_list.append(round(uv_index, 2))

    print(f"📅 Ay: {month}")
    print(f"   • Günlük UV (J/m²/day):       {uv_j:.2f}")
    print(f"   • Ortalama UV gücü (W/m²):    {uv_wm2:.6f}")
    print(f"   • Yaklaşık UV Index:           {uv_index:.2f}\n")


# --- Sonuçları yazdır ---
print("\n📊 2021 Aylık Ortalama Yaklaşık UV Index (Bursa):")
for month, uv in zip(month_names, uv_index_list):
    print(f"   {month}: {uv}")


⏩ Dosya zaten var, indirme atlandı.

🔍 HESAPLAMA ADIMLARI (Ay Ay):


🔍 HESAPLAMA ADIMLARI (Ay Ay):


🔍 HESAPLAMA ADIMLARI (Ay Ay):

📅 Ay: 01
   • Günlük UV (J/m²/day):       767021.06
   • Ortalama UV gücü (W/m²):    8.877559
   • Yaklaşık UV Index:           2.66

📅 Ay: 02
   • Günlük UV (J/m²/day):       1222920.75
   • Ortalama UV gücü (W/m²):    14.154175
   • Yaklaşık UV Index:           4.25

📅 Ay: 03
   • Günlük UV (J/m²/day):       1515842.88
   • Ortalama UV gücü (W/m²):    17.544478
   • Yaklaşık UV Index:           5.79

📅 Ay: 04
   • Günlük UV (J/m²/day):       2031111.88
   • Ortalama UV gücü (W/m²):    23.508239
   • Yaklaşık UV Index:           7.76

📅 Ay: 05
   • Günlük UV (J/m²/day):       2654701.00
   • Ortalama UV gücü (W/m²):    30.725706
   • Yaklaşık UV Index:           10.14

📅 Ay: 06
   • Günlük UV (J/m²/day):       2706584.50
   • Ortalama UV gücü (W/m²):    31.326209
   • Yaklaşık UV Index:           10.96

📅 Ay: 07
   • Günlük UV (J/m²/day):       2853960.75

In [2]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"start_date": "2021-04-01",
	"end_date": "2022-12-31",
	"daily": ["uv_index_clear_sky_max", "sunshine_duration", "uv_index_max"],
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_uv_index_clear_sky_max = daily.Variables(0).ValuesAsNumpy()
daily_sunshine_duration = daily.Variables(1).ValuesAsNumpy()
daily_uv_index_max = daily.Variables(2).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["uv_index_clear_sky_max"] = daily_uv_index_clear_sky_max
daily_data["sunshine_duration"] = daily_sunshine_duration
daily_data["uv_index_max"] = daily_uv_index_max

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)

# İlk 10 satırı göster
print(daily_dataframe.head(10))

# Her sütunda kaç tane boş (NaN) değer var
print("\nEksik veri sayısı:")
print(daily_dataframe.isna().sum())

# Eğer ilk birkaç günün verisi boş mu kontrol etmek istersen
print("\nİlk 10 günün verisi:")
print(daily_dataframe.iloc[:10])


Coordinates: 52.52000045776367°N 13.419998168945312°E
Elevation: 38.0 m asl
Timezone difference to GMT+0: 0s

Daily data
                          date  uv_index_clear_sky_max  sunshine_duration  \
0   2021-04-01 00:00:00+00:00                    4.50       36383.027344   
1   2021-04-02 00:00:00+00:00                    4.60       40598.464844   
2   2021-04-03 00:00:00+00:00                    4.55       42468.386719   
3   2021-04-04 00:00:00+00:00                    4.70       42836.460938   
4   2021-04-05 00:00:00+00:00                    4.55       18894.515625   
..                        ...                     ...                ...   
635 2022-12-27 00:00:00+00:00                    0.95        9004.584961   
636 2022-12-28 00:00:00+00:00                    0.95           0.000000   
637 2022-12-29 00:00:00+00:00                    1.00        1412.500610   
638 2022-12-30 00:00:00+00:00                    1.00       20396.574219   
639 2022-12-31 00:00:00+00:00             

In [18]:
import requests
from datetime import datetime

api_key = "4d23dd53d01b90b2ccc827ad1a43ef24"
lat = 39.9334
lon = 32.8597

# One Call API (Üyelik gerektirir, günlük 1000 çağrı ücretsizdir)
url = f"https://api.openweathermap.org/data/3.0/onecall?lat={lat}&lon={lon}&exclude=minutely,hourly&appid={api_key}&units=metric"

# Not: Bu kodun çalışması için geçerli bir API Key'e ihtiyacınız vardır.
response = requests.get(url)
data = response.json()
current_uv = data['current']['uvi']
print(f"Anlık UV İndeksi: {current_uv}")

KeyError: 'current'

In [3]:
import requests
import pandas as pd
import time
import os
import re

# --- 1. AYARLAR ---
# İstenilen Tarih Aralığı (2020 başı - 2021 4. ay başı)
START_DATE = "2020-01-01"
END_DATE = "2021-03-31" 

# 🔑 BURAYA VISUAL CROSSING API KEY'İNİ GİRMELİSİN
VISUAL_CROSSING_API_KEY = "CKGUWMVXM3PHNVSRQY224XCFN"

# Çıktı Dosya İsmi
FILE_VC_UV = "Rapor_VisualCrossing_UV_2020_2021.xlsx"

# Ham verilerin saklanacağı klasör
CACHE_FOLDER = "sehir_verileri_vc_uv"
os.makedirs(CACHE_FOLDER, exist_ok=True)

# Şehir Listesi (Aynı liste)
ERA5_CITY_AREAS = {
    # 🌍 ABD Eyalet Başkentleri ve Bölgeler
    'Birmingham - Alabama': '33.7700/-87.0600/33.2700/-86.5600',
    'Anchorage - Alaska': '61.4200/-150.1700/60.9200/-149.6700',
    'American Samoa Pago Pago': '-14.0200/-170.9500/-14.5200/-170.4500',
    'Phoenix - Arizona': '33.7400/-112.3500/33.2400/-111.8500',
    'Little Rock - Arkansas': '34.9900/-92.5300/34.4900/-92.0300',
    'Sacramento - California': '38.7700/-121.7500/38.2700/-121.2500',
    'Denver - Colorado': '39.9800/-105.1300/39.4800/-104.6300',
    'Bridgeport - Connecticut': '41.4300/-73.4400/40.9300/-72.9400',
    'Dover - Delaware': '39.3600/-75.7600/38.8600/-75.2600',
    'Washington - District of Columbia': '39.1500/-77.2800/38.6500/-76.7800',
    'Tallahassee - Florida': '30.6500/-84.5500/30.1500/-84.0500',
    'Atlanta - Georgia': '34.0000/-84.6400/33.5000/-84.1400',
    'Guam - Tamuning': '13.7300/144.5700/13.2300/145.0700',
    'Honolulu - Hawaii': '21.5800/-158.0800/21.0800/-157.5800',
    'Boise - Idaho': '43.8700/-116.4500/43.3700/-115.9500',
    'Chicago - Illinois': '41.9700/-87.9700/41.4700/-87.4700',
    'Indianapolis - Indiana': '39.9700/-86.3700/39.4700/-85.8700',
    'Des Moines - Iowa': '41.8800/-93.8200/41.3800/-93.3200',
    'Topeka - Kansas': '39.2600/-95.9600/38.7600/-95.4600',
    'Louisville - Kentucky': '38.4500/-85.9600/37.9500/-85.4600',
    'New Orleans - Louisiana': '30.2000/-90.2000/29.7000/-89.7000',
    'Augusta - Maine': '44.4700/-69.9600/43.9700/-69.4600',
    'Baltimore - Maryland': '39.5400/-76.8600/39.0400/-76.3600',
    'Boston - Massachusetts': '42.6100/-71.3100/42.1100/-70.8100',
    'Lansing - Michigan': '42.9700/-84.7600/42.4700/-84.2600',
    'Minneapolis - Minnesota': '45.2800/-93.4200/44.7800/-92.9200',
    'Jackson - Mississippi': '32.5800/-90.3000/32.0800/-89.8000',
    'Jefferson City - Missouri': '38.7700/-92.3600/38.2700/-91.8600',
    'Helena - Montana': '46.8800/-112.2600/46.3800/-111.7600',
    'Omaha - Nebraska': '41.4800/-96.2200/40.9800/-95.7200',
    'Carson City - Nevada': '39.4100/-120.0200/38.9100/-119.5200',
    'Concord - New Hampshire': '43.4000/-71.7500/42.9000/-71.2500',
    'Newark - New Jersey': '40.9700/-74.3700/40.4700/-73.8700',
    'Santa Fe - New Mexico': '35.9800/-106.1800/35.4800/-105.6800',
    'New York - New York': '40.9600/-74.2500/40.4600/-73.7500',
    'Charlotte - North Carolina': '35.4200/-80.9300/34.9200/-80.4300',
    'Bismarck - North Dakota': '47.0500/-101.0400/46.5500/-100.5400',
    'Saipan, Northern Mariana Islands': '15.4200/145.7200/14.9200/146.2200',
    'Columbus - Ohio': '40.1000/-83.1000/39.6000/-82.6000',
    'Oklahoma City - Oklahoma': '35.7700/-97.7700/35.2700/-97.2700',
    'Salem - Oregon': '45.1900/-123.2800/44.6900/-122.7800',
    'Philadelphia - Pennsylvania': '40.2300/-75.4500/39.7300/-74.9500',
    'San Juan - Puerto Rico': '18.6600/-66.2100/18.1600/-65.7100',
    'Providence - Rhode Island': '41.9200/-71.6000/41.4200/-71.1000',
    'Charleston - South Carolina': '33.0300/-80.1800/32.5300/-79.6800',
    'Columbia - South Dakota': '44.3700/-99.8000/43.8700/-99.3000',
    'Memphis - Tennessee': '35.3200/-90.2700/34.8200/-89.7700',
    'Austin - Texas': '30.5200/-97.9900/30.0200/-97.4900',
    'Salt Lake City - Utah': '40.9800/-112.0200/40.4800/-111.5200',
    'Burlington - Vermont': '44.7300/-73.4600/44.2300/-72.9600',
    'St. John - Virgin Islands': '18.5300/-64.9000/18.0300/-64.4000',
    'Richmond - Virginia': '37.7500/-77.6200/37.2500/-77.1200',
    'Seattle - Washington': '47.8200/-122.5300/47.3200/-122.0300',
    'Charleston - West Virginia': '38.5800/-81.8200/38.0800/-81.3200',
    'Milwaukee - Wisconsin': '43.2000/-88.1000/42.7000/-87.6000',
    'Cheyenne - Wyoming': '41.3800/-105.0200/40.8800/-104.5200',

    # 🌍 Uluslararası Şehirler ve Bölgeler
    'Andorra la Vella - Andorra': '42.7600/1.2700/42.2600/1.7700',
    'St. John\'s, Antigua and Barbuda': '17.3800/-61.8500/16.8800/-61.3500',
    'Oranjestad, Aruba': '12.8000/-70.2800/12.3000/-69.7800',
    'Nassau, The Bahamas': '25.2600/-77.5800/24.7600/-77.0800',
    'Manama - Bahrain': '26.4200/50.4800/25.9200/50.9800',
    'Brussels - Belgium': '51.0500/4.0200/50.5500/4.5200',
    'Belmopan - Belize': '17.5000/-89.0100/17.0000/-88.5100',
    'Road Town - British Virgin Islands': '18.6000/-64.7000/18.1000/-64.2000',
    'Bandar Seri Begawan - Brunei': '5.1400/114.6900/4.6400/115.1900',
    'Praia - Cape Verde': '15.1500/-23.6000/14.6500/-23.1000',
    'Willemstad - Curacao': '12.3700/-69.1000/11.8700/-68.6000',
    'Nicosia - Cyprus': '35.3200/33.1700/34.8200/33.6700',
    'Prag - Czechia': '50.3000/14.1700/49.8000/14.6700',
    'Copenhagen - Denmark': '55.9200/12.3200/55.4200/12.8200',
    'Roseau - Dominica': '15.4200/-61.3700/14.9200/-60.8700',
    'Tallinn - Estonia': '59.6400/24.5100/59.1400/25.0100',
    'Suva - Fiji': '-17.9600/178.2900/-18.4600/178.7900',
    'Helsinki - Finland': '60.4200/24.7500/59.9200/25.2500',
    'Papeete - French Polynesia': '-17.2200/-149.7700/-17.7200/-149.2700',
    'Gibraltar': '36.3200/-5.5500/35.8200/-5.0500',
    'St. George\'s, - Grenada': '12.3200/-61.7700/11.8200/-61.2700',
    'Hong Kong': '22.5700/113.9700/22.0700/114.4700',
    'Reykjavik - Iceland': '64.3000/-22.0300/63.8000/-21.5300',
    'Dublin - Ireland': '53.5800/-6.4000/53.0800/-5.9000',
    'Douglas - Isle of Man': '54.3200/-4.6200/53.8200/-4.1200',
    'Jerusalem - Israel': '32.0700/35.0800/31.5700/35.5800',
    'Kingston - Jamaica': '18.2400/-76.9900/17.7400/-76.4900',
    'Tarawa - Kiribati': '1.6700/172.7800/1.1700/173.2800',
    'Pristina - Kosovo': '42.8700/21.0700/42.3700/21.5700',
    'Kuwait': '29.6200/47.7800/29.1200/48.2800',
    'Riga - Latvia': '57.2000/24.0000/56.7000/24.5000',
    'Vaduz - Liechtenstein': '47.3200/9.3200/46.8200/9.8200',
    'Vilnius - Lithuania': '54.9700/25.0700/54.4700/25.5700',
    'Luxembourg': '49.8600/5.8800/49.3600/6.3800',
    'Macao': '22.4400/113.2900/21.9400/113.7900',
    'Male - Maldives': '4.3200/73.4800/3.8200/73.9800',
    'Valletta - Malta': '35.9400/14.3600/35.4400/14.8600',
    'Majuro - Marshall Islands': '7.3800/171.1800/6.8800/171.6800',
    'Port Louis - Mauritius': '-19.9600/57.3600/-20.4600/57.8600',
    'Weno - Micronesia': '7.6000/151.7800/7.1000/152.2800',
    'Monaco': '43.9800/7.1700/43.4800/7.6700',
    'Amsterdam - Netherlands': '52.6200/4.6500/52.1200/5.1500',
    'Nouméa - New Caledonia': '-22.0300/166.2000/-22.5300/166.7000',
    'Oslo - Norway': '60.1000/10.3700/59.6000/10.8700',
    'Jerusalem - Palestine': '32.0700/35.0800/31.5700/35.5800',
    'Doha - Qatar': '25.5700/51.3800/25.0700/51.8800',
    'Basseterre - Saint Kitts and Nevis': '17.4200/-62.8200/16.9200/-62.3200',
    'Castries - Saint Lucia': '14.2200/-61.1200/13.7200/-60.6200',
    'Kingstown - Saint Vincent and the Grenadines': '13.3200/-61.3200/12.8200/-60.8200',
    'Apia - Samoa': '-13.7700/-172.0800/-14.2700/-171.5800',
    'San Marino': '44.0700/12.3200/43.5700/12.8200',
    'Victoria - Seychelles': '-4.3200/55.3700/-4.8200/55.8700',
    'Singapore': '1.5700/103.7700/1.0700/104.2700',
    'Bratislava - Slovakia': '48.3700/16.8200/47.8700/17.3200',
    'Ljubljana - Slovenia': '46.2200/14.3200/45.7200/14.8200',
    'Honiara - Solomon Islands': '-9.2700/159.8700/-9.7700/160.3700',
    'Stockholm - Sweden': '59.5700/17.8200/59.0700/18.3200',
    'Zurich - Switzerland': '47.5700/8.3700/47.0700/8.8700',
    'New Taipei City - Taiwan': '25.3200/121.3200/24.8200/121.8200',
    'Dili - Timor': '-8.3700/125.4200/-8.8700/125.9200',
    'Nuku\'alofa - Tonga': '-21.0700/-175.3200/-21.5700/-174.8200',
    'Port of Spain - Trinidad and Tobago': '10.8700/-61.6200/10.3700/-61.1200',
    'Abu Dhabi - United Arab Emirates': '24.7700/54.1200/24.2700/54.6200'
}

# --- YARDIMCI FONKSİYONLAR ---

def parse_coordinates_to_centroid(coord_string):
    try:
        parts = [float(x) for x in coord_string.split('/')]
        lat1, lon1, lat2, lon2 = parts
        center_lat = (lat1 + lat2) / 2
        center_lon = (lon1 + lon2) / 2
        return center_lat, center_lon
    except:
        return None, None

def get_safe_filename(city_name):
    # Dosya isminde sorun çıkarabilecek karakterleri temizle
    safe_name = re.sub(r'[^\w\s-]', '', city_name).strip().replace(' ', '_')
    return os.path.join(CACHE_FOLDER, f"{safe_name}.csv")

def fetch_visual_crossing_data(city_name, lat, lon):
    if VISUAL_CROSSING_API_KEY == "BURAYA_API_KEY_YAZILACAK":
        print("\n❌ HATA: Lütfen kodun başındaki VISUAL_CROSSING_API_KEY değişkenine API anahtarını girin!")
        return None

    # Visual Crossing URL Yapısı: /timeline/[lat,lon]/[start]/[end]
    base_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline"
    request_url = f"{base_url}/{lat},{lon}/{START_DATE}/{END_DATE}"
    
    params = {
        "unitGroup": "metric", # Santigrat ve metrik birimler
        "key": VISUAL_CROSSING_API_KEY,
        "include": "days",     # Sadece günlük veriler
        "elements": "datetime,uvindex", # Sadece tarih ve UV index
        "contentType": "json"
    }
    
    print(f"📡 VC'den veri çekiliyor ({city_name})...", end="\r")
    
    try:
        response = requests.get(request_url, params=params)
        
        if response.status_code == 429:
            print(f"\n⛔ KOTA AŞILDI! İşlem durduruluyor. ({city_name})")
            return "QUOTA_EXCEEDED"
            
        response.raise_for_status() 
        return response.json()
    except Exception as e:
        print(f"\n❌ Hata ({city_name}): {e}")
        return None

# --- ANA İŞLEM ---

uv_data_list = []

print(f"⏳ Visual Crossing UV İşlemi Başlıyor... ({START_DATE} - {END_DATE})")
print(f"⚠️ UYARI: Bu işlem API kotanızı tüketir!")
print(f"📂 Veriler '{CACHE_FOLDER}' klasörüne kaydedilecek.")
print(f"🌍 Toplam Şehir Sayısı: {len(ERA5_CITY_AREAS)}\n")

for city, coords in ERA5_CITY_AREAS.items():
    
    # 1. Koordinatları al
    lat, lon = parse_coordinates_to_centroid(coords)
    if lat is None:
        print(f"⚠️ Koordinat hatası: {city}")
        continue
    
    # 2. Dosya kontrolü (Varsa indirmez)
    file_path = get_safe_filename(city)
    df = None
    
    if os.path.exists(file_path):
        print(f"📁 Dosyadan okunuyor: {city}" + " " * 30, end="\r")
        try:
            df = pd.read_csv(file_path)
            df["date"] = pd.to_datetime(df["date"])
        except Exception as e:
            print(f"\n⚠️ Dosya okuma hatası ({city}): {e}")
            continue
            
    else:
        # Dosya yoksa API'den çek
        data = fetch_visual_crossing_data(city, lat, lon)
        
        if data == "QUOTA_EXCEEDED":
            break # Döngüyü tamamen kır, daha fazla deneme.
        
        if data and "days" in data:
            # JSON yapısını DataFrame'e çevir
            records = []
            for day in data["days"]:
                records.append({
                    "date": day.get("datetime"),
                    "uv_val": day.get("uvindex")
                })
            
            df = pd.DataFrame(records)
            df["date"] = pd.to_datetime(df["date"])
            
            # Veriyi temizle ve CSV'ye kaydet
            df["uv_val"] = pd.to_numeric(df["uv_val"], errors='coerce').fillna(0)
            df.to_csv(file_path, index=False) 
            
            # API nezaket beklemesi
            time.sleep(1)
        else:
            print(f"\n⚠️ API'den veri dönmedi veya hata: {city}")
            continue

    # 3. Hesaplama 
    if df is not None:
        # --- AYLIK HESAPLAMA (ORTALAMA) ---
        monthly_uv = df.resample('ME', on='date')["uv_val"].mean()
        
        uv_row = {"Sehir": city}
        
        for date_idx, val in monthly_uv.items():
            col_name = date_idx.strftime("%Y-%m")
            uv_row[col_name] = round(val, 2)
            
        uv_data_list.append(uv_row)

print("\n\n💾 Veriler işleniyor ve Excel'e kaydediliyor...")

if uv_data_list:
    df_uv_final = pd.DataFrame(uv_data_list)
    df_uv_final.to_excel(FILE_VC_UV, index=False)
    print(f"✅ Visual Crossing UV Dosyası Hazır: {FILE_VC_UV}")
else:
    print("⚠️ Kaydedilecek veri bulunamadı.")

print("\n🏁 Tüm işlemler tamamlandı.")

⏳ Visual Crossing UV İşlemi Başlıyor... (2020-01-01 - 2021-03-31)
⚠️ UYARI: Bu işlem API kotanızı tüketir!
📂 Veriler 'sehir_verileri_vc_uv' klasörüne kaydedilecek.
🌍 Toplam Şehir Sayısı: 119

📡 VC'den veri çekiliyor (Birmingham - Alabama)...
⛔ KOTA AŞILDI! İşlem durduruluyor. (Birmingham - Alabama)


💾 Veriler işleniyor ve Excel'e kaydediliyor...
⚠️ Kaydedilecek veri bulunamadı.

🏁 Tüm işlemler tamamlandı.


In [1]:
import requests
import pandas as pd
import time
import os
import re

# --- 1. AYARLAR ---
# Tarih Formatı: YYYYMMDD
START_DATE = "20200101"
END_DATE = "20221231"

# Çıktı Dosya İsmi
FILE_NASA_MAX = "Rapor_NASA_UV_Index_Gunluk_Max.xlsx"

# Klasör (Saatlik veri büyük olduğu için ayrı klasör yapalım)
CACHE_FOLDER = "sehir_verileri_nasa_hourly"
os.makedirs(CACHE_FOLDER, exist_ok=True)

# Şehir Listesi
ERA5_CITY_AREAS = {
    # 🌍 ABD Eyalet Başkentleri ve Bölgeler
    'Birmingham - Alabama': '33.7700/-87.0600/33.2700/-86.5600',
    'Anchorage - Alaska': '61.4200/-150.1700/60.9200/-149.6700',
    'American Samoa Pago Pago': '-14.0200/-170.9500/-14.5200/-170.4500',
    'Phoenix - Arizona': '33.7400/-112.3500/33.2400/-111.8500',
    'Little Rock - Arkansas': '34.9900/-92.5300/34.4900/-92.0300',
    'Sacramento - California': '38.7700/-121.7500/38.2700/-121.2500',
    'Denver - Colorado': '39.9800/-105.1300/39.4800/-104.6300',
    'Bridgeport - Connecticut': '41.4300/-73.4400/40.9300/-72.9400',
    'Dover - Delaware': '39.3600/-75.7600/38.8600/-75.2600',
    'Washington - District of Columbia': '39.1500/-77.2800/38.6500/-76.7800',
    'Tallahassee - Florida': '30.6500/-84.5500/30.1500/-84.0500',
    'Atlanta - Georgia': '34.0000/-84.6400/33.5000/-84.1400',
    'Guam - Tamuning': '13.7300/144.5700/13.2300/145.0700',
    'Honolulu - Hawaii': '21.5800/-158.0800/21.0800/-157.5800',
    'Boise - Idaho': '43.8700/-116.4500/43.3700/-115.9500',
    'Chicago - Illinois': '41.9700/-87.9700/41.4700/-87.4700',
    'Indianapolis - Indiana': '39.9700/-86.3700/39.4700/-85.8700',
    'Des Moines - Iowa': '41.8800/-93.8200/41.3800/-93.3200',
    'Topeka - Kansas': '39.2600/-95.9600/38.7600/-95.4600',
    'Louisville - Kentucky': '38.4500/-85.9600/37.9500/-85.4600',
    'New Orleans - Louisiana': '30.2000/-90.2000/29.7000/-89.7000',
    'Augusta - Maine': '44.4700/-69.9600/43.9700/-69.4600',
    'Baltimore - Maryland': '39.5400/-76.8600/39.0400/-76.3600',
    'Boston - Massachusetts': '42.6100/-71.3100/42.1100/-70.8100',
    'Lansing - Michigan': '42.9700/-84.7600/42.4700/-84.2600',
    'Minneapolis - Minnesota': '45.2800/-93.4200/44.7800/-92.9200',
    'Jackson - Mississippi': '32.5800/-90.3000/32.0800/-89.8000',
    'Jefferson City - Missouri': '38.7700/-92.3600/38.2700/-91.8600',
    'Helena - Montana': '46.8800/-112.2600/46.3800/-111.7600',
    'Omaha - Nebraska': '41.4800/-96.2200/40.9800/-95.7200',
    'Carson City - Nevada': '39.4100/-120.0200/38.9100/-119.5200',
    'Concord - New Hampshire': '43.4000/-71.7500/42.9000/-71.2500',
    'Newark - New Jersey': '40.9700/-74.3700/40.4700/-73.8700',
    'Santa Fe - New Mexico': '35.9800/-106.1800/35.4800/-105.6800',
    'New York - New York': '40.9600/-74.2500/40.4600/-73.7500',
    'Charlotte - North Carolina': '35.4200/-80.9300/34.9200/-80.4300',
    'Bismarck - North Dakota': '47.0500/-101.0400/46.5500/-100.5400',
    'Saipan, Northern Mariana Islands': '15.4200/145.7200/14.9200/146.2200',
    'Columbus - Ohio': '40.1000/-83.1000/39.6000/-82.6000',
    'Oklahoma City - Oklahoma': '35.7700/-97.7700/35.2700/-97.2700',
    'Salem - Oregon': '45.1900/-123.2800/44.6900/-122.7800',
    'Philadelphia - Pennsylvania': '40.2300/-75.4500/39.7300/-74.9500',
    'San Juan - Puerto Rico': '18.6600/-66.2100/18.1600/-65.7100',
    'Providence - Rhode Island': '41.9200/-71.6000/41.4200/-71.1000',
    'Charleston - South Carolina': '33.0300/-80.1800/32.5300/-79.6800',
    'Columbia - South Dakota': '44.3700/-99.8000/43.8700/-99.3000',
    'Memphis - Tennessee': '35.3200/-90.2700/34.8200/-89.7700',
    'Austin - Texas': '30.5200/-97.9900/30.0200/-97.4900',
    'Salt Lake City - Utah': '40.9800/-112.0200/40.4800/-111.5200',
    'Burlington - Vermont': '44.7300/-73.4600/44.2300/-72.9600',
    'St. John - Virgin Islands': '18.5300/-64.9000/18.0300/-64.4000',
    'Richmond - Virginia': '37.7500/-77.6200/37.2500/-77.1200',
    'Seattle - Washington': '47.8200/-122.5300/47.3200/-122.0300',
    'Charleston - West Virginia': '38.5800/-81.8200/38.0800/-81.3200',
    'Milwaukee - Wisconsin': '43.2000/-88.1000/42.7000/-87.6000',
    'Cheyenne - Wyoming': '41.3800/-105.0200/40.8800/-104.5200',

    # 🌍 Uluslararası Şehirler ve Bölgeler
    'Andorra la Vella - Andorra': '42.7600/1.2700/42.2600/1.7700',
    'St. John\'s, Antigua and Barbuda': '17.3800/-61.8500/16.8800/-61.3500',
    'Oranjestad, Aruba': '12.8000/-70.2800/12.3000/-69.7800',
    'Nassau, The Bahamas': '25.2600/-77.5800/24.7600/-77.0800',
    'Manama - Bahrain': '26.4200/50.4800/25.9200/50.9800',
    'Brussels - Belgium': '51.0500/4.0200/50.5500/4.5200',
    'Belmopan - Belize': '17.5000/-89.0100/17.0000/-88.5100',
    'Road Town - British Virgin Islands': '18.6000/-64.7000/18.1000/-64.2000',
    'Bandar Seri Begawan - Brunei': '5.1400/114.6900/4.6400/115.1900',
    'Praia - Cape Verde': '15.1500/-23.6000/14.6500/-23.1000',
    'Willemstad - Curacao': '12.3700/-69.1000/11.8700/-68.6000',
    'Nicosia - Cyprus': '35.3200/33.1700/34.8200/33.6700',
    'Prag - Czechia': '50.3000/14.1700/49.8000/14.6700',
    'Copenhagen - Denmark': '55.9200/12.3200/55.4200/12.8200',
    'Roseau - Dominica': '15.4200/-61.3700/14.9200/-60.8700',
    'Tallinn - Estonia': '59.6400/24.5100/59.1400/25.0100',
    'Suva - Fiji': '-17.9600/178.2900/-18.4600/178.7900',
    'Helsinki - Finland': '60.4200/24.7500/59.9200/25.2500',
    'Papeete - French Polynesia': '-17.2200/-149.7700/-17.7200/-149.2700',
    'Gibraltar': '36.3200/-5.5500/35.8200/-5.0500',
    'St. George\'s, - Grenada': '12.3200/-61.7700/11.8200/-61.2700',
    'Hong Kong': '22.5700/113.9700/22.0700/114.4700',
    'Reykjavik - Iceland': '64.3000/-22.0300/63.8000/-21.5300',
    'Dublin - Ireland': '53.5800/-6.4000/53.0800/-5.9000',
    'Douglas - Isle of Man': '54.3200/-4.6200/53.8200/-4.1200',
    'Jerusalem - Israel': '32.0700/35.0800/31.5700/35.5800',
    'Kingston - Jamaica': '18.2400/-76.9900/17.7400/-76.4900',
    'Tarawa - Kiribati': '1.6700/172.7800/1.1700/173.2800',
    'Pristina - Kosovo': '42.8700/21.0700/42.3700/21.5700',
    'Kuwait': '29.6200/47.7800/29.1200/48.2800',
    'Riga - Latvia': '57.2000/24.0000/56.7000/24.5000',
    'Vaduz - Liechtenstein': '47.3200/9.3200/46.8200/9.8200',
    'Vilnius - Lithuania': '54.9700/25.0700/54.4700/25.5700',
    'Luxembourg': '49.8600/5.8800/49.3600/6.3800',
    'Macao': '22.4400/113.2900/21.9400/113.7900',
    'Male - Maldives': '4.3200/73.4800/3.8200/73.9800',
    'Valletta - Malta': '35.9400/14.3600/35.4400/14.8600',
    'Majuro - Marshall Islands': '7.3800/171.1800/6.8800/171.6800',
    'Port Louis - Mauritius': '-19.9600/57.3600/-20.4600/57.8600',
    'Weno - Micronesia': '7.6000/151.7800/7.1000/152.2800',
    'Monaco': '43.9800/7.1700/43.4800/7.6700',
    'Amsterdam - Netherlands': '52.6200/4.6500/52.1200/5.1500',
    'Nouméa - New Caledonia': '-22.0300/166.2000/-22.5300/166.7000',
    'Oslo - Norway': '60.1000/10.3700/59.6000/10.8700',
    'Jerusalem - Palestine': '32.0700/35.0800/31.5700/35.5800',
    'Doha - Qatar': '25.5700/51.3800/25.0700/51.8800',
    'Basseterre - Saint Kitts and Nevis': '17.4200/-62.8200/16.9200/-62.3200',
    'Castries - Saint Lucia': '14.2200/-61.1200/13.7200/-60.6200',
    'Kingstown - Saint Vincent and the Grenadines': '13.3200/-61.3200/12.8200/-60.8200',
    'Apia - Samoa': '-13.7700/-172.0800/-14.2700/-171.5800',
    'San Marino': '44.0700/12.3200/43.5700/12.8200',
    'Victoria - Seychelles': '-4.3200/55.3700/-4.8200/55.8700',
    'Singapore': '1.5700/103.7700/1.0700/104.2700',
    'Bratislava - Slovakia': '48.3700/16.8200/47.8700/17.3200',
    'Ljubljana - Slovenia': '46.2200/14.3200/45.7200/14.8200',
    'Honiara - Solomon Islands': '-9.2700/159.8700/-9.7700/160.3700',
    'Stockholm - Sweden': '59.5700/17.8200/59.0700/18.3200',
    'Zurich - Switzerland': '47.5700/8.3700/47.0700/8.8700',
    'New Taipei City - Taiwan': '25.3200/121.3200/24.8200/121.8200',
    'Dili - Timor': '-8.3700/125.4200/-8.8700/125.9200',
    'Nuku\'alofa - Tonga': '-21.0700/-175.3200/-21.5700/-174.8200',
    'Port of Spain - Trinidad and Tobago': '10.8700/-61.6200/10.3700/-61.1200',
    'Abu Dhabi - United Arab Emirates': '24.7700/54.1200/24.2700/54.6200'
}

# --- YARDIMCI FONKSİYONLAR ---

def parse_coordinates_to_centroid(coord_string):
    try:
        parts = [float(x) for x in coord_string.split('/')]
        center_lat = (parts[0] + parts[2]) / 2
        center_lon = (parts[1] + parts[3]) / 2
        return center_lat, center_lon
    except:
        return None, None

def get_safe_filename(city_name):
    safe_name = re.sub(r'[^\w\s-]', '', city_name).strip().replace(' ', '_')
    return os.path.join(CACHE_FOLDER, f"{safe_name}.csv")

def fetch_nasa_hourly_uv(city_name, lat, lon):
    # 🌟 NASA POWER SAATLİK API
    url = "https://power.larc.nasa.gov/api/temporal/hourly/point"
    
    params = {
        "parameters": "ALLSKY_SFC_UV_INDEX", 
        "community": "RE", 
        "longitude": lon,
        "latitude": lat,
        "start": START_DATE,
        "end": END_DATE,
        "format": "JSON"
    }
    
    print(f"📡 NASA Saatlik Veri İndiriliyor ({city_name}) - Bu işlem sürebilir...", end="\r")
    try:
        # Saatlik veri büyüktür, timeout'u artırıyoruz
        response = requests.get(url, params=params, timeout=60)
        response.raise_for_status() 
        return response.json()
    except Exception as e:
        print(f"\n❌ Hata ({city_name}): {e}")
        return None

# --- ANA İŞLEM ---

uv_data_list = []

print(f"⏳ NASA Saatlik Veri -> Günlük MAX İşlemi Başlıyor... ({START_DATE} - {END_DATE})")
print("ℹ️  Bu işlem, her saatin verisini indirip o günün en yüksek değerini bulur.")
print("ℹ️  Veri boyutu büyük olduğu için her şehir için 3-5 saniye sürebilir.\n")

for city, coords in ERA5_CITY_AREAS.items():
    
    lat, lon = parse_coordinates_to_centroid(coords)
    if lat is None: continue
    
    file_path = get_safe_filename(city)
    df = None
    
    # 2. Dosya Kontrolü
    if os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path)
            df["date"] = pd.to_datetime(df["date"])
        except:
            pass
            
    # Dosya yoksa API'den çek
    if df is None:
        data = fetch_nasa_hourly_uv(city, lat, lon)
        
        if data and "properties" in data:
            try:
                # NASA Saatlik Formatı: YYYYMMDDHH
                uv_dict = data['properties']['parameter']['ALLSKY_SFC_UV_INDEX']
                
                df = pd.DataFrame(list(uv_dict.items()), columns=['datetime_str', 'uv_val'])
                
                # Tarih/Saat Formatını Düzelt
                df['date_full'] = pd.to_datetime(df['datetime_str'], format='%Y%m%d%H')
                
                # -999 değerlerini temizle
                df = df[df['uv_val'] != -999]
                
                # Sadece Tarih Kısmını Al (Saat bilgisini at, gruplamak için)
                df['date'] = df['date_full'].dt.date
                df['date'] = pd.to_datetime(df['date']) # Tekrar datetime objesi yap
                
                # CSV'ye kaydet (Bütün saatlik veriyi saklayalım)
                df.to_csv(file_path, index=False)
                
                time.sleep(2) # NASA'ya yüklenmemek için bekleme
                
            except KeyError:
                print(f"\n⚠️ Veri formatı hatası: {city}")
                continue
        else:
            print(f"\n⚠️ Veri dönmedi: {city}")
            continue

    # 3. HESAPLAMA ZİNCİRİ
    if df is not None:
        # A) Her Günün MAKSİMUM değerini bul (Öğle saati zirvesi)
        daily_max = df.groupby('date')['uv_val'].max()
        
        # B) Bu maksimumların AYLIK ORTALAMASINI al
        # (Yani: O ay boyunca öğlenleri ortalama kaç UV gördük?)
        monthly_uv = daily_max.resample('ME').mean()
        
        uv_row = {"Sehir": city}
        for date_idx, val in monthly_uv.items():
            col_name = date_idx.strftime("%Y-%m")
            uv_row[col_name] = round(val, 2)
            
        uv_data_list.append(uv_row)

print("\n\n💾 Veriler işleniyor ve Excel'e kaydediliyor...")

if uv_data_list:
    df_final = pd.DataFrame(uv_data_list)
    df_final.to_excel(FILE_NASA_MAX, index=False)
    print(f"✅ NASA Raporu Hazır: {FILE_NASA_MAX}")
else:
    print("❌ Kaydedilecek veri bulunamadı.")

print("\n🏁 Tüm işlemler tamamlandı.")

⏳ NASA Saatlik Veri -> Günlük MAX İşlemi Başlıyor... (20200101 - 20221231)
ℹ️  Bu işlem, her saatin verisini indirip o günün en yüksek değerini bulur.
ℹ️  Veri boyutu büyük olduğu için her şehir için 3-5 saniye sürebilir.

📡 NASA Saatlik Veri İndiriliyor (Abu Dhabi - United Arab Emirates) - Bu işlem sürebilir......ebilir...

💾 Veriler işleniyor ve Excel'e kaydediliyor...
✅ NASA Raporu Hazır: Rapor_NASA_UV_Index_Gunluk_Max.xlsx

🏁 Tüm işlemler tamamlandı.


In [2]:
import requests
import pandas as pd
import time
import os
import re

# --- 1. AYARLAR ---
# Tarih Formatı: YYYYMMDD
START_DATE = "20200101"
# 🌟 DÜZELTME: Bitiş tarihini 15 Ocak 2023 yaptık.
# NASA API bu tarihe kadar olan veriyi çeker.
END_DATE = "20230115" 

# Çıktı Dosya İsmi
FILE_NASA_MAX = "Rapor_NASA_UV_Index_Gunluk_Max_37Ay.xlsx"

# Klasör
CACHE_FOLDER = "sehir_verileri_nasa_hourly_37ay"
os.makedirs(CACHE_FOLDER, exist_ok=True)

# Şehir Listesi (Aynı liste korunuyor)
ERA5_CITY_AREAS = {
    # 🌍 ABD Eyalet Başkentleri ve Bölgeler
    'Birmingham - Alabama': '33.7700/-87.0600/33.2700/-86.5600',
    'Anchorage - Alaska': '61.4200/-150.1700/60.9200/-149.6700',
    'American Samoa Pago Pago': '-14.0200/-170.9500/-14.5200/-170.4500',
    'Phoenix - Arizona': '33.7400/-112.3500/33.2400/-111.8500',
    'Little Rock - Arkansas': '34.9900/-92.5300/34.4900/-92.0300',
    'Sacramento - California': '38.7700/-121.7500/38.2700/-121.2500',
    'Denver - Colorado': '39.9800/-105.1300/39.4800/-104.6300',
    'Bridgeport - Connecticut': '41.4300/-73.4400/40.9300/-72.9400',
    'Dover - Delaware': '39.3600/-75.7600/38.8600/-75.2600',
    'Washington - District of Columbia': '39.1500/-77.2800/38.6500/-76.7800',
    'Tallahassee - Florida': '30.6500/-84.5500/30.1500/-84.0500',
    'Atlanta - Georgia': '34.0000/-84.6400/33.5000/-84.1400',
    'Guam - Tamuning': '13.7300/144.5700/13.2300/145.0700',
    'Honolulu - Hawaii': '21.5800/-158.0800/21.0800/-157.5800',
    'Boise - Idaho': '43.8700/-116.4500/43.3700/-115.9500',
    'Chicago - Illinois': '41.9700/-87.9700/41.4700/-87.4700',
    'Indianapolis - Indiana': '39.9700/-86.3700/39.4700/-85.8700',
    'Des Moines - Iowa': '41.8800/-93.8200/41.3800/-93.3200',
    'Topeka - Kansas': '39.2600/-95.9600/38.7600/-95.4600',
    'Louisville - Kentucky': '38.4500/-85.9600/37.9500/-85.4600',
    'New Orleans - Louisiana': '30.2000/-90.2000/29.7000/-89.7000',
    'Augusta - Maine': '44.4700/-69.9600/43.9700/-69.4600',
    'Baltimore - Maryland': '39.5400/-76.8600/39.0400/-76.3600',
    'Boston - Massachusetts': '42.6100/-71.3100/42.1100/-70.8100',
    'Lansing - Michigan': '42.9700/-84.7600/42.4700/-84.2600',
    'Minneapolis - Minnesota': '45.2800/-93.4200/44.7800/-92.9200',
    'Jackson - Mississippi': '32.5800/-90.3000/32.0800/-89.8000',
    'Jefferson City - Missouri': '38.7700/-92.3600/38.2700/-91.8600',
    'Helena - Montana': '46.8800/-112.2600/46.3800/-111.7600',
    'Omaha - Nebraska': '41.4800/-96.2200/40.9800/-95.7200',
    'Carson City - Nevada': '39.4100/-120.0200/38.9100/-119.5200',
    'Concord - New Hampshire': '43.4000/-71.7500/42.9000/-71.2500',
    'Newark - New Jersey': '40.9700/-74.3700/40.4700/-73.8700',
    'Santa Fe - New Mexico': '35.9800/-106.1800/35.4800/-105.6800',
    'New York - New York': '40.9600/-74.2500/40.4600/-73.7500',
    'Charlotte - North Carolina': '35.4200/-80.9300/34.9200/-80.4300',
    'Bismarck - North Dakota': '47.0500/-101.0400/46.5500/-100.5400',
    'Saipan, Northern Mariana Islands': '15.4200/145.7200/14.9200/146.2200',
    'Columbus - Ohio': '40.1000/-83.1000/39.6000/-82.6000',
    'Oklahoma City - Oklahoma': '35.7700/-97.7700/35.2700/-97.2700',
    'Salem - Oregon': '45.1900/-123.2800/44.6900/-122.7800',
    'Philadelphia - Pennsylvania': '40.2300/-75.4500/39.7300/-74.9500',
    'San Juan - Puerto Rico': '18.6600/-66.2100/18.1600/-65.7100',
    'Providence - Rhode Island': '41.9200/-71.6000/41.4200/-71.1000',
    'Charleston - South Carolina': '33.0300/-80.1800/32.5300/-79.6800',
    'Columbia - South Dakota': '44.3700/-99.8000/43.8700/-99.3000',
    'Memphis - Tennessee': '35.3200/-90.2700/34.8200/-89.7700',
    'Austin - Texas': '30.5200/-97.9900/30.0200/-97.4900',
    'Salt Lake City - Utah': '40.9800/-112.0200/40.4800/-111.5200',
    'Burlington - Vermont': '44.7300/-73.4600/44.2300/-72.9600',
    'St. John - Virgin Islands': '18.5300/-64.9000/18.0300/-64.4000',
    'Richmond - Virginia': '37.7500/-77.6200/37.2500/-77.1200',
    'Seattle - Washington': '47.8200/-122.5300/47.3200/-122.0300',
    'Charleston - West Virginia': '38.5800/-81.8200/38.0800/-81.3200',
    'Milwaukee - Wisconsin': '43.2000/-88.1000/42.7000/-87.6000',
    'Cheyenne - Wyoming': '41.3800/-105.0200/40.8800/-104.5200',

    # 🌍 Uluslararası Şehirler ve Bölgeler
    'Andorra la Vella - Andorra': '42.7600/1.2700/42.2600/1.7700',
    'St. John\'s, Antigua and Barbuda': '17.3800/-61.8500/16.8800/-61.3500',
    'Oranjestad, Aruba': '12.8000/-70.2800/12.3000/-69.7800',
    'Nassau, The Bahamas': '25.2600/-77.5800/24.7600/-77.0800',
    'Manama - Bahrain': '26.4200/50.4800/25.9200/50.9800',
    'Brussels - Belgium': '51.0500/4.0200/50.5500/4.5200',
    'Belmopan - Belize': '17.5000/-89.0100/17.0000/-88.5100',
    'Road Town - British Virgin Islands': '18.6000/-64.7000/18.1000/-64.2000',
    'Bandar Seri Begawan - Brunei': '5.1400/114.6900/4.6400/115.1900',
    'Praia - Cape Verde': '15.1500/-23.6000/14.6500/-23.1000',
    'Willemstad - Curacao': '12.3700/-69.1000/11.8700/-68.6000',
    'Nicosia - Cyprus': '35.3200/33.1700/34.8200/33.6700',
    'Prag - Czechia': '50.3000/14.1700/49.8000/14.6700',
    'Copenhagen - Denmark': '55.9200/12.3200/55.4200/12.8200',
    'Roseau - Dominica': '15.4200/-61.3700/14.9200/-60.8700',
    'Tallinn - Estonia': '59.6400/24.5100/59.1400/25.0100',
    'Suva - Fiji': '-17.9600/178.2900/-18.4600/178.7900',
    'Helsinki - Finland': '60.4200/24.7500/59.9200/25.2500',
    'Papeete - French Polynesia': '-17.2200/-149.7700/-17.7200/-149.2700',
    'Gibraltar': '36.3200/-5.5500/35.8200/-5.0500',
    'St. George\'s, - Grenada': '12.3200/-61.7700/11.8200/-61.2700',
    'Hong Kong': '22.5700/113.9700/22.0700/114.4700',
    'Reykjavik - Iceland': '64.3000/-22.0300/63.8000/-21.5300',
    'Dublin - Ireland': '53.5800/-6.4000/53.0800/-5.9000',
    'Douglas - Isle of Man': '54.3200/-4.6200/53.8200/-4.1200',
    'Jerusalem - Israel': '32.0700/35.0800/31.5700/35.5800',
    'Kingston - Jamaica': '18.2400/-76.9900/17.7400/-76.4900',
    'Tarawa - Kiribati': '1.6700/172.7800/1.1700/173.2800',
    'Pristina - Kosovo': '42.8700/21.0700/42.3700/21.5700',
    'Kuwait': '29.6200/47.7800/29.1200/48.2800',
    'Riga - Latvia': '57.2000/24.0000/56.7000/24.5000',
    'Vaduz - Liechtenstein': '47.3200/9.3200/46.8200/9.8200',
    'Vilnius - Lithuania': '54.9700/25.0700/54.4700/25.5700',
    'Luxembourg': '49.8600/5.8800/49.3600/6.3800',
    'Macao': '22.4400/113.2900/21.9400/113.7900',
    'Male - Maldives': '4.3200/73.4800/3.8200/73.9800',
    'Valletta - Malta': '35.9400/14.3600/35.4400/14.8600',
    'Majuro - Marshall Islands': '7.3800/171.1800/6.8800/171.6800',
    'Port Louis - Mauritius': '-19.9600/57.3600/-20.4600/57.8600',
    'Weno - Micronesia': '7.6000/151.7800/7.1000/152.2800',
    'Monaco': '43.9800/7.1700/43.4800/7.6700',
    'Amsterdam - Netherlands': '52.6200/4.6500/52.1200/5.1500',
    'Nouméa - New Caledonia': '-22.0300/166.2000/-22.5300/166.7000',
    'Oslo - Norway': '60.1000/10.3700/59.6000/10.8700',
    'Jerusalem - Palestine': '32.0700/35.0800/31.5700/35.5800',
    'Doha - Qatar': '25.5700/51.3800/25.0700/51.8800',
    'Basseterre - Saint Kitts and Nevis': '17.4200/-62.8200/16.9200/-62.3200',
    'Castries - Saint Lucia': '14.2200/-61.1200/13.7200/-60.6200',
    'Kingstown - Saint Vincent and the Grenadines': '13.3200/-61.3200/12.8200/-60.8200',
    'Apia - Samoa': '-13.7700/-172.0800/-14.2700/-171.5800',
    'San Marino': '44.0700/12.3200/43.5700/12.8200',
    'Victoria - Seychelles': '-4.3200/55.3700/-4.8200/55.8700',
    'Singapore': '1.5700/103.7700/1.0700/104.2700',
    'Bratislava - Slovakia': '48.3700/16.8200/47.8700/17.3200',
    'Ljubljana - Slovenia': '46.2200/14.3200/45.7200/14.8200',
    'Honiara - Solomon Islands': '-9.2700/159.8700/-9.7700/160.3700',
    'Stockholm - Sweden': '59.5700/17.8200/59.0700/18.3200',
    'Zurich - Switzerland': '47.5700/8.3700/47.0700/8.8700',
    'New Taipei City - Taiwan': '25.3200/121.3200/24.8200/121.8200',
    'Dili - Timor': '-8.3700/125.4200/-8.8700/125.9200',
    'Nuku\'alofa - Tonga': '-21.0700/-175.3200/-21.5700/-174.8200',
    'Port of Spain - Trinidad and Tobago': '10.8700/-61.6200/10.3700/-61.1200',
    'Abu Dhabi - United Arab Emirates': '24.7700/54.1200/24.2700/54.6200'
}

# --- YARDIMCI FONKSİYONLAR ---

def parse_coordinates_to_centroid(coord_string):
    try:
        parts = [float(x) for x in coord_string.split('/')]
        center_lat = (parts[0] + parts[2]) / 2
        center_lon = (parts[1] + parts[3]) / 2
        return center_lat, center_lon
    except:
        return None, None

def get_safe_filename(city_name):
    safe_name = re.sub(r'[^\w\s-]', '', city_name).strip().replace(' ', '_')
    return os.path.join(CACHE_FOLDER, f"{safe_name}.csv")

def fetch_nasa_hourly_uv(city_name, lat, lon):
    url = "https://power.larc.nasa.gov/api/temporal/hourly/point"
    params = {
        "parameters": "ALLSKY_SFC_UV_INDEX", 
        "community": "RE", 
        "longitude": lon,
        "latitude": lat,
        "start": START_DATE,
        "end": END_DATE, # ⚠️ 20230115 olarak ayarlandı
        "format": "JSON"
    }
    print(f"📡 NASA Saatlik Veri İndiriliyor ({city_name}) - Bu işlem sürebilir...", end="\r")
    try:
        # Timeout süresini biraz daha artırıyoruz (veri büyüdü)
        response = requests.get(url, params=params, timeout=90)
        response.raise_for_status() 
        return response.json()
    except Exception as e:
        print(f"\n❌ Hata ({city_name}): {e}")
        return None

# --- ANA İŞLEM ---

uv_data_list = []

print(f"⏳ NASA Saatlik Veri -> Günlük MAX İşlemi Başlıyor... ({START_DATE} - {END_DATE})")
print("ℹ️  Veri seti 37 ayı (Ocak 2020 - 15 Ocak 2023) kapsayacak.\n")

for city, coords in ERA5_CITY_AREAS.items():
    
    lat, lon = parse_coordinates_to_centroid(coords)
    if lat is None: continue
    
    file_path = get_safe_filename(city)
    df = None
    
    # Dosya Kontrolü (Varsa indirmez)
    if os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path)
            df["date"] = pd.to_datetime(df["date"])
        except:
            pass
            
    # Dosya yoksa API'den çek
    if df is None:
        data = fetch_nasa_hourly_uv(city, lat, lon)
        
        if data and "properties" in data:
            try:
                # NASA Saatlik Formatı: YYYYMMDDHH
                uv_dict = data['properties']['parameter']['ALLSKY_SFC_UV_INDEX']
                
                df = pd.DataFrame(list(uv_dict.items()), columns=['datetime_str', 'uv_val'])
                
                # Tarih/Saat Formatını Düzelt
                df['date_full'] = pd.to_datetime(df['datetime_str'], format='%Y%m%d%H')
                
                # -999 değerlerini temizle
                df = df[df['uv_val'] != -999]
                
                # Sadece Tarih Kısmını Al (Saat bilgisini at, gruplamak için)
                df['date'] = df['date_full'].dt.date
                df['date'] = pd.to_datetime(df['date']) 
                
                # CSV'ye kaydet
                df.to_csv(file_path, index=False)
                
                time.sleep(2) 
                
            except KeyError:
                print(f"\n⚠️ Veri formatı hatası: {city}")
                continue
        else:
            print(f"\n⚠️ Veri dönmedi: {city}")
            continue

    # 3. HESAPLAMA ZİNCİRİ
    if df is not None:
        # A) Her Günün MAKSİMUM değerini bul
        daily_max = df.groupby('date')['uv_val'].max()
        
        # B) Bu maksimumların AYLIK ORTALAMASINI al
        # 'ME' (Month End) kullanıyoruz. 2023-01-15'e kadar veri olduğu için,
        # Pandas bunu "2023-01-31" (Ocak sonu) etiketi altında, 
        # sadece elindeki 15 günün ortalamasını alarak hesaplar.
        monthly_uv = daily_max.resample('ME').mean()
        
        uv_row = {"Sehir": city}
        for date_idx, val in monthly_uv.items():
            col_name = date_idx.strftime("%Y-%m")
            uv_row[col_name] = round(val, 2)
            
        uv_data_list.append(uv_row)

print("\n\n💾 Veriler işleniyor ve Excel'e kaydediliyor...")

if uv_data_list:
    df_final = pd.DataFrame(uv_data_list)
    
    # İsteğe bağlı: Sütun sayısını kontrol et
    print(f"Toplam Ay Sayısı: {len(df_final.columns) - 1}") # -1 (Şehir sütunu hariç)
    
    df_final.to_excel(FILE_NASA_MAX, index=False)
    print(f"✅ NASA Raporu Hazır: {FILE_NASA_MAX}")
else:
    print("❌ Kaydedilecek veri bulunamadı.")

print("\n🏁 Tüm işlemler tamamlandı.")

⏳ NASA Saatlik Veri -> Günlük MAX İşlemi Başlıyor... (20200101 - 20230115)
ℹ️  Veri seti 37 ayı (Ocak 2020 - 15 Ocak 2023) kapsayacak.

📡 NASA Saatlik Veri İndiriliyor (Abu Dhabi - United Arab Emirates) - Bu işlem sürebilir......ebilir...

💾 Veriler işleniyor ve Excel'e kaydediliyor...
Toplam Ay Sayısı: 37
✅ NASA Raporu Hazır: Rapor_NASA_UV_Index_Gunluk_Max_37Ay.xlsx

🏁 Tüm işlemler tamamlandı.


In [1]:
import requests
import pandas as pd
import time
import os
import re

# --- 1. AYARLAR ---
# NASA API Tarih Formatı: YYYYMMDD
START_DATE = "19900101"
END_DATE = "19930115" 

# Çıktı Dosya İsmi (Günlük Max UV)
FILE_NASA_DAILY = "Rapor_NASA_Gunluk_UV_Index_Max_1111Gun_old.xlsx"

# Klasör (Günlük veriler için yeni klasör)
CACHE_FOLDER = "sehir_verileri_nasa_uv_gunluk_old"
os.makedirs(CACHE_FOLDER, exist_ok=True)

# Şehir Listesi (Aynı liste)
ERA5_CITY_AREAS = {
    # 🌍 ABD Eyalet Başkentleri ve Bölgeler
    'Birmingham - Alabama': '33.7700/-87.0600/33.2700/-86.5600',
    'Anchorage - Alaska': '61.4200/-150.1700/60.9200/-149.6700',
    'American Samoa Pago Pago': '-14.0200/-170.9500/-14.5200/-170.4500',
    'Phoenix - Arizona': '33.7400/-112.3500/33.2400/-111.8500',
    'Little Rock - Arkansas': '34.9900/-92.5300/34.4900/-92.0300',
    'Sacramento - California': '38.7700/-121.7500/38.2700/-121.2500',
    'Denver - Colorado': '39.9800/-105.1300/39.4800/-104.6300',
    'Bridgeport - Connecticut': '41.4300/-73.4400/40.9300/-72.9400',
    'Dover - Delaware': '39.3600/-75.7600/38.8600/-75.2600',
    'Washington - District of Columbia': '39.1500/-77.2800/38.6500/-76.7800',
    'Tallahassee - Florida': '30.6500/-84.5500/30.1500/-84.0500',
    'Atlanta - Georgia': '34.0000/-84.6400/33.5000/-84.1400',
    'Guam - Tamuning': '13.7300/144.5700/13.2300/145.0700',
    'Honolulu - Hawaii': '21.5800/-158.0800/21.0800/-157.5800',
    'Boise - Idaho': '43.8700/-116.4500/43.3700/-115.9500',
    'Chicago - Illinois': '41.9700/-87.9700/41.4700/-87.4700',
    'Indianapolis - Indiana': '39.9700/-86.3700/39.4700/-85.8700',
    'Des Moines - Iowa': '41.8800/-93.8200/41.3800/-93.3200',
    'Topeka - Kansas': '39.2600/-95.9600/38.7600/-95.4600',
    'Louisville - Kentucky': '38.4500/-85.9600/37.9500/-85.4600',
    'New Orleans - Louisiana': '30.2000/-90.2000/29.7000/-89.7000',
    'Augusta - Maine': '44.4700/-69.9600/43.9700/-69.4600',
    'Baltimore - Maryland': '39.5400/-76.8600/39.0400/-76.3600',
    'Boston - Massachusetts': '42.6100/-71.3100/42.1100/-70.8100',
    'Lansing - Michigan': '42.9700/-84.7600/42.4700/-84.2600',
    'Minneapolis - Minnesota': '45.2800/-93.4200/44.7800/-92.9200',
    'Jackson - Mississippi': '32.5800/-90.3000/32.0800/-89.8000',
    'Jefferson City - Missouri': '38.7700/-92.3600/38.2700/-91.8600',
    'Helena - Montana': '46.8800/-112.2600/46.3800/-111.7600',
    'Omaha - Nebraska': '41.4800/-96.2200/40.9800/-95.7200',
    'Carson City - Nevada': '39.4100/-120.0200/38.9100/-119.5200',
    'Concord - New Hampshire': '43.4000/-71.7500/42.9000/-71.2500',
    'Newark - New Jersey': '40.9700/-74.3700/40.4700/-73.8700',
    'Santa Fe - New Mexico': '35.9800/-106.1800/35.4800/-105.6800',
    'New York - New York': '40.9600/-74.2500/40.4600/-73.7500',
    'Charlotte - North Carolina': '35.4200/-80.9300/34.9200/-80.4300',
    'Bismarck - North Dakota': '47.0500/-101.0400/46.5500/-100.5400',
    'Saipan, Northern Mariana Islands': '15.4200/145.7200/14.9200/146.2200',
    'Columbus - Ohio': '40.1000/-83.1000/39.6000/-82.6000',
    'Oklahoma City - Oklahoma': '35.7700/-97.7700/35.2700/-97.2700',
    'Salem - Oregon': '45.1900/-123.2800/44.6900/-122.7800',
    'Philadelphia - Pennsylvania': '40.2300/-75.4500/39.7300/-74.9500',
    'San Juan - Puerto Rico': '18.6600/-66.2100/18.1600/-65.7100',
    'Providence - Rhode Island': '41.9200/-71.6000/41.4200/-71.1000',
    'Charleston - South Carolina': '33.0300/-80.1800/32.5300/-79.6800',
    'Columbia - South Dakota': '44.3700/-99.8000/43.8700/-99.3000',
    'Memphis - Tennessee': '35.3200/-90.2700/34.8200/-89.7700',
    'Austin - Texas': '30.5200/-97.9900/30.0200/-97.4900',
    'Salt Lake City - Utah': '40.9800/-112.0200/40.4800/-111.5200',
    'Burlington - Vermont': '44.7300/-73.4600/44.2300/-72.9600',
    'St. John - Virgin Islands': '18.5300/-64.9000/18.0300/-64.4000',
    'Richmond - Virginia': '37.7500/-77.6200/37.2500/-77.1200',
    'Seattle - Washington': '47.8200/-122.5300/47.3200/-122.0300',
    'Charleston - West Virginia': '38.5800/-81.8200/38.0800/-81.3200',
    'Milwaukee - Wisconsin': '43.2000/-88.1000/42.7000/-87.6000',
    'Cheyenne - Wyoming': '41.3800/-105.0200/40.8800/-104.5200',
    'Andorra la Vella - Andorra': '42.7600/1.2700/42.2600/1.7700',
    'St. John\'s, Antigua and Barbuda': '17.3800/-61.8500/16.8800/-61.3500',
    'Oranjestad, Aruba': '12.8000/-70.2800/12.3000/-69.7800',
    'Nassau, The Bahamas': '25.2600/-77.5800/24.7600/-77.0800',
    'Manama - Bahrain': '26.4200/50.4800/25.9200/50.9800',
    'Brussels - Belgium': '51.0500/4.0200/50.5500/4.5200',
    'Belmopan - Belize': '17.5000/-89.0100/17.0000/-88.5100',
    'Road Town - British Virgin Islands': '18.6000/-64.7000/18.1000/-64.2000',
    'Bandar Seri Begawan - Brunei': '5.1400/114.6900/4.6400/115.1900',
    'Praia - Cape Verde': '15.1500/-23.6000/14.6500/-23.1000',
    'Willemstad - Curacao': '12.3700/-69.1000/11.8700/-68.6000',
    'Nicosia - Cyprus': '35.3200/33.1700/34.8200/33.6700',
    'Prag - Czechia': '50.3000/14.1700/49.8000/14.6700',
    'Copenhagen - Denmark': '55.9200/12.3200/55.4200/12.8200',
    'Roseau - Dominica': '15.4200/-61.3700/14.9200/-60.8700',
    'Tallinn - Estonia': '59.6400/24.5100/59.1400/25.0100',
    'Suva - Fiji': '-17.9600/178.2900/-18.4600/178.7900',
    'Helsinki - Finland': '60.4200/24.7500/59.9200/25.2500',
    'Papeete - French Polynesia': '-17.2200/-149.7700/-17.7200/-149.2700',
    'Gibraltar': '36.3200/-5.5500/35.8200/-5.0500',
    'St. George\'s, - Grenada': '12.3200/-61.7700/11.8200/-61.2700',
    'Hong Kong': '22.5700/113.9700/22.0700/114.4700',
    'Reykjavik - Iceland': '64.3000/-22.0300/63.8000/-21.5300',
    'Dublin - Ireland': '53.5800/-6.4000/53.0800/-5.9000',
    'Douglas - Isle of Man': '54.3200/-4.6200/53.8200/-4.1200',
    'Jerusalem - Israel': '32.0700/35.0800/31.5700/35.5800',
    'Kingston - Jamaica': '18.2400/-76.9900/17.7400/-76.4900',
    'Tarawa - Kiribati': '1.6700/172.7800/1.1700/173.2800',
    'Pristina - Kosovo': '42.8700/21.0700/42.3700/21.5700',
    'Kuwait': '29.6200/47.7800/29.1200/48.2800',
    'Riga - Latvia': '57.2000/24.0000/56.7000/24.5000',
    'Vaduz - Liechtenstein': '47.3200/9.3200/46.8200/9.8200',
    'Vilnius - Lithuania': '54.9700/25.0700/54.4700/25.5700',
    'Luxembourg': '49.8600/5.8800/49.3600/6.3800',
    'Macao': '22.4400/113.2900/21.9400/113.7900',
    'Male - Maldives': '4.3200/73.4800/3.8200/73.9800',
    'Valletta - Malta': '35.9400/14.3600/35.4400/14.8600',
    'Majuro - Marshall Islands': '7.3800/171.1800/6.8800/171.6800',
    'Port Louis - Mauritius': '-19.9600/57.3600/-20.4600/57.8600',
    'Weno - Micronesia': '7.6000/151.7800/7.1000/152.2800',
    'Monaco': '43.9800/7.1700/43.4800/7.6700',
    'Amsterdam - Netherlands': '52.6200/4.6500/52.1200/5.1500',
    'Nouméa - New Caledonia': '-22.0300/166.2000/-22.5300/166.7000',
    'Oslo - Norway': '60.1000/10.3700/59.6000/10.8700',
    'Jerusalem - Palestine': '32.0700/35.0800/31.5700/35.5800',
    'Doha - Qatar': '25.5700/51.3800/25.0700/51.8800',
    'Basseterre - Saint Kitts and Nevis': '17.4200/-62.8200/16.9200/-62.3200',
    'Castries - Saint Lucia': '14.2200/-61.1200/13.7200/-60.6200',
    'Kingstown - Saint Vincent and the Grenadines': '13.3200/-61.3200/12.8200/-60.8200',
    'Apia - Samoa': '-13.7700/-172.0800/-14.2700/-171.5800',
    'San Marino': '44.0700/12.3200/43.5700/12.8200',
    'Victoria - Seychelles': '-4.3200/55.3700/-4.8200/55.8700',
    'Singapore': '1.5700/103.7700/1.0700/104.2700',
    'Bratislava - Slovakia': '48.3700/16.8200/47.8700/17.3200',
    'Ljubljana - Slovenia': '46.2200/14.3200/45.7200/14.8200',
    'Honiara - Solomon Islands': '-9.2700/159.8700/-9.7700/160.3700',
    'Stockholm - Sweden': '59.5700/17.8200/59.0700/18.3200',
    'Zurich - Switzerland': '47.5700/8.3700/47.0700/8.8700',
    'New Taipei City - Taiwan': '25.3200/121.3200/24.8200/121.8200',
    'Dili - Timor': '-8.3700/125.4200/-8.8700/125.9200',
    'Nuku\'alofa - Tonga': '-21.0700/-175.3200/-21.5700/-174.8200',
    'Port of Spain - Trinidad and Tobago': '10.8700/-61.6200/10.3700/-61.1200',
    'Abu Dhabi - United Arab Emirates': '24.7700/54.1200/24.2700/54.6200'
}

# --- YARDIMCI FONKSİYONLAR ---

def parse_coordinates_to_centroid(coord_string):
    try:
        parts = [float(x) for x in coord_string.split('/')]
        center_lat = (parts[0] + parts[2]) / 2
        center_lon = (parts[1] + parts[3]) / 2
        return center_lat, center_lon
    except:
        return None, None

def get_safe_filename(city_name):
    safe_name = re.sub(r'[^\w\s-]', '', city_name).strip().replace(' ', '_')
    return os.path.join(CACHE_FOLDER, f"{safe_name}.csv")

def fetch_nasa_hourly_uv(city_name, lat, lon):
    url = "https://power.larc.nasa.gov/api/temporal/hourly/point"
    params = {
        "parameters": "ALLSKY_SFC_UV_INDEX", 
        "community": "RE", 
        "longitude": lon,
        "latitude": lat,
        "start": START_DATE,
        "end": END_DATE, 
        "format": "JSON"
    }
    print(f"📡 NASA Saatlik Veri İndiriliyor ({city_name}) - Bu işlem sürebilir...", end="\r")
    try:
        # Timeout süresini yüksek tutuyoruz çünkü NASA 3 yıllık saatlik veriyi geç dönebiliyor
        response = requests.get(url, params=params, timeout=90)
        response.raise_for_status() 
        return response.json()
    except Exception as e:
        print(f"\n❌ Hata ({city_name}): {e}")
        return None

# --- ANA İŞLEM ---

# Verileri sütun bazlı (dictionary key = Şehir İsmi) saklayacağız
all_cities_data = {} 

print(f"⏳ NASA Saatlik -> Günlük MAX İşlemi Başlıyor... ({START_DATE} - {END_DATE})")
print(f"📂 Veriler '{CACHE_FOLDER}' klasörüne önbelleğe alınacak.")

for city, coords in ERA5_CITY_AREAS.items():
    
    lat, lon = parse_coordinates_to_centroid(coords)
    if lat is None: continue
    
    file_path = get_safe_filename(city)
    df = None
    
    # Dosya Kontrolü (Varsa indirmez)
    if os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path)
            # CSV'den okuyunca string gelir, datetime'a çevir
            if 'date_full' in df.columns:
                df["date_full"] = pd.to_datetime(df["date_full"])
        except:
            pass
            
    # Dosya yoksa API'den çek
    if df is None:
        data = fetch_nasa_hourly_uv(city, lat, lon)
        
        if data and "properties" in data:
            try:
                # NASA Saatlik Formatı: YYYYMMDDHH (Properties içindeki key'ler)
                uv_dict = data['properties']['parameter']['ALLSKY_SFC_UV_INDEX']
                
                df = pd.DataFrame(list(uv_dict.items()), columns=['datetime_str', 'uv_val'])
                
                # Tarih/Saat Formatını Düzelt
                df['date_full'] = pd.to_datetime(df['datetime_str'], format='%Y%m%d%H')
                
                # -999 değerlerini temizle (NASA'da hata kodu -999'dur)
                df = df[df['uv_val'] != -999]
                
                # Veriyi Temizle ve Kaydet
                df.to_csv(file_path, index=False)
                
                time.sleep(2) # Nezaket beklemesi
                
            except KeyError:
                print(f"\n⚠️ Veri formatı hatası: {city}")
                continue
        else:
            print(f"\n⚠️ Veri dönmedi: {city}")
            continue

    # 3. GÜNLÜK MAX HESAPLAMA ve SÖZLÜĞE EKLEME
    if df is not None:
        # Sadece Tarih Kısmını Al (Saat bilgisini at)
        # Bu kısım önemlidir: Saatlik veriyi güne indirger.
        df['date'] = df['date_full'].dt.date
        df['date'] = pd.to_datetime(df['date']) # Datetime objesi olduğundan emin ol
        
        # Güneşlenme veya Sıcaklık kodundaki mantık:
        # groupby ile her günün en büyük (MAX) değerini al
        daily_max_series = df.groupby('date')['uv_val'].max()
        
        # Seriyi sözlüğe ekle (Key: Şehir Adı)
        all_cities_data[city] = daily_max_series

print("\n\n💾 Veriler birleştiriliyor ve Excel'e kaydediliyor...")

if all_cities_data:
    # 1. Dictionary'den DataFrame oluştur
    df_temp = pd.DataFrame(all_cities_data)
    
    # 2. Transpose al (Satırlar: Şehirler, Sütunlar: Tarihler)
    df_final = df_temp.T
    
    # 3. Sütun isimlerini string formatına çevir (YYYY-MM-DD)
    df_final.columns = [col.strftime('%Y-%m-%d') for col in df_final.columns]
    
    # 4. Şehir ismini sütun yap
    df_final.reset_index(inplace=True)
    df_final.rename(columns={'index': 'Sehir'}, inplace=True)

    # 5. Kaydet
    df_final.to_excel(FILE_NASA_DAILY, index=False)
    
    print(f"✅ NASA Günlük UV Dosyası Hazır: {FILE_NASA_DAILY}")
    print(f"ℹ️  Toplam Gün (Sütun) Sayısı: {df_final.shape[1] - 1}") 
else:
    print("❌ Kaydedilecek veri bulunamadı.")

print("\n🏁 Tüm işlemler tamamlandı.")

⏳ NASA Saatlik -> Günlük MAX İşlemi Başlıyor... (19900101 - 19930115)
📂 Veriler 'sehir_verileri_nasa_uv_gunluk_old' klasörüne önbelleğe alınacak.
📡 NASA Saatlik Veri İndiriliyor (Birmingham - Alabama) - Bu işlem sürebilir...
❌ Hata (Birmingham - Alabama): 422 Client Error: Unprocessable Entity for url: https://power.larc.nasa.gov/api/temporal/hourly/point?parameters=ALLSKY_SFC_UV_INDEX&community=RE&longitude=-86.81&latitude=33.52&start=19900101&end=19930115&format=JSON

⚠️ Veri dönmedi: Birmingham - Alabama
📡 NASA Saatlik Veri İndiriliyor (Anchorage - Alaska) - Bu işlem sürebilir...
❌ Hata (Anchorage - Alaska): 422 Client Error: Unprocessable Entity for url: https://power.larc.nasa.gov/api/temporal/hourly/point?parameters=ALLSKY_SFC_UV_INDEX&community=RE&longitude=-149.92&latitude=61.17&start=19900101&end=19930115&format=JSON

⚠️ Veri dönmedi: Anchorage - Alaska
📡 NASA Saatlik Veri İndiriliyor (American Samoa Pago Pago) - Bu işlem sürebilir...
❌ Hata (American Samoa Pago Pago): 422 Cli

KeyboardInterrupt: 

In [2]:
import requests
import pandas as pd
import time
import os
import re

# --- 1. AYARLAR ---
# Tarih Formatı: YYYYMMDD
START_DATE = "20200101"
# Bitiş Tarihi: 15 Ocak 2023
END_DATE = "20230115" 

# Çıktı Dosya İsmi (GÜNLÜK OLARAK GÜNCELLENDİ)
FILE_NASA_DAILY = "Rapor_NASA_UV_Index_Gunluk_Max_Tam_Liste.xlsx"

# Klasör
CACHE_FOLDER = "sehir_verileri_nasa_hourly_37ay"
os.makedirs(CACHE_FOLDER, exist_ok=True)

# Şehir Listesi
ERA5_CITY_AREAS = {
    # --- ABD Eyalet Başkentleri ve Bölgeler ---
    'Birmingham - Alabama': '33.7700/-87.0600/33.2700/-86.5600',
    'Anchorage - Alaska': '61.4200/-150.1700/60.9200/-149.6700',
    'American Samoa - Pago Pago': '-14.0200/-170.9500/-14.5200/-170.4500',
    'Phoenix - Arizona': '33.7400/-112.3500/33.2400/-111.8500',
    'Little Rock - Arkansas': '34.9900/-92.5300/34.4900/-92.0300',
    'Sacramento - California': '38.7700/-121.7500/38.2700/-121.2500',
    'Denver - Colorado': '39.9800/-105.1300/39.4800/-104.6300',
    'Bridgeport - Connecticut': '41.4300/-73.4400/40.9300/-72.9400',
    'Dover - Delaware': '39.3600/-75.7600/38.8600/-75.2600',
    'Washington - District of Columbia': '39.1500/-77.2800/38.6500/-76.7800',
    'Tallahassee - Florida': '30.6500/-84.5500/30.1500/-84.0500',
    'Atlanta - Georgia': '34.0000/-84.6400/33.5000/-84.1400',
    'Guam': '13.7300/144.5700/13.2300/145.0700',
    'Honolulu - Hawaii': '21.5800/-158.0800/21.0800/-157.5800',
    'Boise - Idaho': '43.8700/-116.4500/43.3700/-115.9500',
    'Chicago - Illinois': '41.9700/-87.9700/41.4700/-87.4700',
    'Indianapolis - Indiana': '39.9700/-86.3700/39.4700/-85.8700',
    'Des Moines - Iowa': '41.8800/-93.8200/41.3800/-93.3200',
    'Topeka - Kansas': '39.2600/-95.9600/38.7600/-95.4600',
    'Frankfort - Kentucky': '38.4500/-85.1200/37.9500/-84.6200',
    'New Orleans - Louisiana': '30.2000/-90.2000/29.7000/-89.7000',
    'Augusta - Maine': '44.5600/-69.9600/44.0600/-69.4600',
    'Baltimore - Maryland': '39.5400/-76.8600/39.0400/-76.3600',
    'Boston - Massachusetts': '42.6100/-71.3100/42.1100/-70.8100',
    'Lansing - Michigan': '42.9700/-84.7600/42.4700/-84.2600',
    'Saint Paul - Minnesota': '45.1900/-93.3500/44.6900/-92.8500',
    'Jackson - Mississippi': '32.5800/-90.3000/32.0800/-89.8000',
    'Jefferson City - Missouri': '38.7700/-92.3600/38.2700/-91.8600',
    'Helena - Montana': '46.8800/-112.2600/46.3800/-111.7600',
    'Omaha - Nebraska': '41.4800/-96.2200/40.9800/-95.7200',
    'Carson City - Nevada': '39.4100/-120.0200/38.9100/-119.5200',
    'Concord - New Hampshire': '43.4000/-71.7500/42.9000/-71.2500',
    'Newark - New Jersey': '40.9700/-74.3700/40.4700/-73.8700',
    'Santa Fe - New Mexico': '35.9800/-106.1800/35.4800/-105.6800',
    'New York - New York': '40.9600/-74.2500/40.4600/-73.7500',
    'Charlotte - North Carolina': '35.4200/-80.9300/34.9200/-80.4300',
    'Bismarck - North Dakota': '47.0500/-101.0400/46.5500/-100.5400',
    'Saipan, Northern Mariana Islands': '15.4200/145.7200/14.9200/146.2200',
    'Columbus - Ohio': '40.1000/-83.1000/39.6000/-82.6000',
    'Oklahoma City - Oklahoma': '35.7700/-97.7700/35.2700/-97.2700',
    'Salem - Oregon': '45.1900/-123.2800/44.6900/-122.7800',
    'Philadelphia - Pennsylvania': '40.2300/-75.4500/39.7300/-74.9500',
    'San Juan - Puerto Rico': '18.6600/-66.2100/18.1600/-65.7100',
    'Providence - Rhode Island': '41.9200/-71.6000/41.4200/-71.1000',
    'Charleston AFB - South Carolina': '33.1400/-80.2900/32.6400/-79.7900',
    'Columbia - South Dakota': '45.8500/-98.5500/45.3500/-98.0500',
    'Memphis - Tennessee': '35.3200/-90.2700/34.8200/-89.7700',
    'Austin - Texas': '30.5200/-97.9900/30.0200/-97.4900',
    'Salt Lake City - Utah': '40.9800/-112.0200/40.4800/-111.5200',
    'Burlington - Vermont': '44.7300/-73.4600/44.2300/-72.9600',
    'Virgin Islands': '18.5800/-65.1500/18.0800/-64.6500',
    'Richmond - Virginia': '37.7500/-77.6200/37.2500/-77.1200',
    'Seattle - Washington': '47.8200/-122.5300/47.3200/-122.0300',
    'Charleston - West Virginia': '38.5800/-81.8200/38.0800/-81.3200',
    'Milwaukee - Wisconsin': '43.2000/-88.1000/42.7000/-87.6000',
    'Cheyenne - Wyoming': '41.3800/-105.0200/40.8800/-104.5200',

    # --- Uluslararası Şehirler ve Bölgeler ---
    'Tirana - Albania': '41.5700/19.5600/41.0700/20.0600',
    'Andorra': '42.7600/1.2700/42.2600/1.7700',
    'St. John\'s, Antigua and Barbuda': '17.3700/-62.0800/16.8700/-61.5800',
    'Oranjestad, Aruba': '12.8000/-70.2800/12.3000/-69.7800',
    'Vienna - Austria': '48.4500/16.1200/47.9500/16.6200',
    'Nassau, The Bahamas': '25.2600/-77.5800/24.7600/-77.0800',
    'Manama - Bahrain': '26.4200/50.4800/25.9200/50.9800',
    'Brussels - Belgium': '51.0500/4.0200/50.5500/4.5200',
    'Belmopan - Belize': '17.5000/-89.0100/17.0000/-88.5100',
    'Sarajevo - Bosnia and Herzegovina': '44.1000/18.1600/43.6000/18.6600',
    'Road Town - British Virgin Islands': '18.6700/-64.8700/18.1700/-64.3700',
    'Bandar Seri Begawan - Brunei': '5.1400/114.6900/4.6400/115.1900',
    'Praia - Cape Verde': '15.1500/-23.6000/14.6500/-23.1000',
    'Zagreb - Croatia': '46.0600/15.7300/45.5600/16.2300',
    'Willemstad - Curacao': '12.3700/-69.1000/11.8700/-68.6000',
    'Nicosia - Cyprus': '35.3200/33.1700/34.8200/33.6700',
    'Prague - Czechia': '50.3000/14.1700/49.8000/14.6700',
    'Copenhagen - Denmark': '55.9200/12.3200/55.4200/12.8200',
    'Roseau - Dominica': '15.5500/-61.6300/15.0500/-61.1300',
    'San Salvador - El Salvador': '13.9400/-89.4700/13.4400/-88.9700',
    'Tallinn - Estonia': '59.6400/24.5100/59.1400/25.0100',
    'Suva - Fiji': '-17.9600/178.2900/-18.4600/178.7900',
    'Helsinki - Finland': '60.4200/24.7500/59.9200/25.2500',
    'Papeete - French Polynesia': '-17.2200/-149.7700/-17.7200/-149.2700',
    'Gibraltar': '36.3200/-5.5500/35.8200/-5.0500',
    'St. George\'s - Grenada': '12.3000/-61.9500/11.8000/-61.4500',
    'Port-au-Prince - Haiti': '18.7900/-72.5800/18.2900/-72.0800',
    'Hong Kong': '22.5700/113.9700/22.0700/114.4700',
    'Budapest - Hungary': '47.7400/18.7900/47.2400/19.2900',
    'Reykjavik - Iceland': '64.3000/-22.0300/63.8000/-21.5300',
    'Dublin - Ireland': '53.5800/-6.4000/53.0800/-5.9000',
    'Douglas - Isle of Man': '54.3200/-4.6200/53.8200/-4.1200',
    'Jerusalem - Israel': '32.0700/35.0800/31.5700/35.5800',
    'Kingston - Jamaica': '18.2400/-76.9900/17.7400/-76.4900',
    'Tarawa - Kiribati': '1.6700/172.7800/1.1700/173.2800',
    'Pristina - Kosovo': '42.9100/20.9100/42.4100/21.4100',
    'Kuwait': '29.6200/47.7800/29.1200/48.2800',
    'Riga - Latvia': '57.2000/24.0000/56.7000/24.5000',
    'Vaduz - Liechtenstein': '47.3200/9.3200/46.8200/9.8200',
    'Vilnius - Lithuania': '54.9700/25.0700/54.4700/25.5700',
    'Luxembourg': '49.8600/5.8800/49.3600/6.3800',
    'Macao': '22.4400/113.2900/21.9400/113.7900',
    'Male - Maldives': '4.3200/73.4800/3.8200/73.9800',
    'Valletta - Malta': '35.9400/14.3600/35.4400/14.8600',
    'Majuro - Marshall Islands': '7.3800/171.1800/6.8800/171.6800',
    'Port Louis - Mauritius': '-19.9600/57.3600/-20.4600/57.8600',
    'Weno - Micronesia': '7.6000/151.7800/7.1000/152.2800',
    'Monaco': '43.9800/7.1700/43.4800/7.6700',
    'Amsterdam - Netherlands': '52.6200/4.6500/52.1200/5.1500',
    'Nouméa - New Caledonia': '-22.0300/166.2000/-22.5300/166.7000',
    'Skopje - North Macedonia': '42.2500/21.1800/41.7500/21.6800',
    'Oslo - Norway': '60.1000/10.3700/59.6000/10.8700',
    'Jerusalem - Palestine': '32.0700/35.0800/31.5700/35.5800',
    'Doha - Qatar': '25.5700/51.3800/25.0700/51.8800',
    'Basseterre - Saint Kitts and Nevis': '17.5500/-62.9700/17.0500/-62.4700',
    'Castries - Saint Lucia': '14.2600/-61.2400/13.7600/-60.7400',
    'Kingstown - Saint Vincent and the Grenadines': '13.4100/-61.4700/12.9100/-60.9700',
    'Apia - Samoa': '-13.7700/-172.0800/-14.2700/-171.5800',
    'San Marino': '44.0700/12.3200/43.5700/12.8200',
    'Belgrade - Serbia': '45.0300/20.1900/44.5300/20.6900',
    'Victoria - Seychelles': '-4.3700/55.2000/-4.8700/55.7000',
    'Singapore': '1.5700/103.7700/1.0700/104.2700',
    'Bratislava - Slovakia': '48.3700/16.8200/47.8700/17.3200',
    'Ljubljana - Slovenia': '46.2200/14.3200/45.7200/14.8200',
    'Honiara - Solomon Islands': '-9.2700/159.8700/-9.7700/160.3700',
    'Stockholm - Sweden': '59.5700/17.8200/59.0700/18.3200',
    'Zurich - Switzerland': '47.5700/8.3700/47.0700/8.8700',
    'New Taipei City - Taiwan': '25.3200/121.3200/24.8200/121.8200',
    'Dili - Timor': '-8.3700/125.4200/-8.8700/125.9200',
    'Nuku\'alofa - Tonga': '-21.0700/-175.3200/-21.5700/-174.8200',
    'Port of Spain - Trinidad and Tobago': '10.8700/-61.6200/10.3700/-61.1200',
    'Abu Dhabi - United Arab Emirates': '24.7700/54.1200/24.2700/54.6200',
    'Montevideo - Uruguay': '-34.6500/-56.4100/-35.1500/-55.9100'
}

# --- YARDIMCI FONKSİYONLAR ---

def parse_coordinates_to_centroid(coord_string):
    try:
        parts = [float(x) for x in coord_string.split('/')]
        center_lat = (parts[0] + parts[2]) / 2
        center_lon = (parts[1] + parts[3]) / 2
        return center_lat, center_lon
    except:
        return None, None

def get_safe_filename(city_name):
    safe_name = re.sub(r'[^\w\s-]', '', city_name).strip().replace(' ', '_')
    return os.path.join(CACHE_FOLDER, f"{safe_name}.csv")

def fetch_nasa_hourly_uv(city_name, lat, lon):
    url = "https://power.larc.nasa.gov/api/temporal/hourly/point"
    params = {
        "parameters": "ALLSKY_SFC_UV_INDEX", 
        "community": "RE", 
        "longitude": lon,
        "latitude": lat,
        "start": START_DATE,
        "end": END_DATE, 
        "format": "JSON"
    }
    print(f"📡 NASA Saatlik Veri İndiriliyor ({city_name}) - Bu işlem sürebilir...", end="\r")
    try:
        response = requests.get(url, params=params, timeout=90)
        response.raise_for_status() 
        return response.json()
    except Exception as e:
        print(f"\n❌ Hata ({city_name}): {e}")
        return None

# --- ANA İŞLEM ---

uv_data_list = []

print(f"⏳ NASA Saatlik Veri -> Günlük MAX İşlemi Başlıyor... ({START_DATE} - {END_DATE})")
print("ℹ️  Amaç: Her günün en yüksek UV değerini (öğle vakti) yakalamak.")
print(f"📂 Veriler '{CACHE_FOLDER}' klasörüne kaydedilecek.\n")

for city, coords in ERA5_CITY_AREAS.items():
    
    lat, lon = parse_coordinates_to_centroid(coords)
    if lat is None: continue
    
    file_path = get_safe_filename(city)
    df = None
    
    # Dosya Kontrolü (Varsa indirmez)
    if os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path)
            df["date"] = pd.to_datetime(df["date"])
        except:
            pass
            
    # Dosya yoksa API'den çek
    if df is None:
        data = fetch_nasa_hourly_uv(city, lat, lon)
        
        if data and "properties" in data:
            try:
                # NASA Saatlik Formatı: YYYYMMDDHH
                uv_dict = data['properties']['parameter']['ALLSKY_SFC_UV_INDEX']
                
                df = pd.DataFrame(list(uv_dict.items()), columns=['datetime_str', 'uv_val'])
                
                # Tarih/Saat Formatını Düzelt
                df['date_full'] = pd.to_datetime(df['datetime_str'], format='%Y%m%d%H')
                
                # -999 değerlerini temizle
                df = df[df['uv_val'] != -999]
                
                # Sadece Tarih Kısmını Al (Saat bilgisini at, gruplamak için)
                df['date'] = df['date_full'].dt.date
                df['date'] = pd.to_datetime(df['date']) 
                
                # CSV'ye kaydet
                df.to_csv(file_path, index=False)
                
                time.sleep(2) 
                
            except KeyError:
                print(f"\n⚠️ Veri formatı hatası: {city}")
                continue
        else:
            print(f"\n⚠️ Veri dönmedi: {city}")
            continue

    # 3. GÜNLÜK MAX HESAPLAMA (DEĞİŞTİRİLEN KISIM)
    if df is not None:
        # A) Her Günün MAKSİMUM değerini bul (Öğle vakti piki)
        daily_max = df.groupby('date')['uv_val'].max()
        
        # B) ARTIK AYLIK ORTALAMA ALMIYORUZ.
        # Doğrudan günleri sütun yapıyoruz.
        
        uv_row = {"Sehir": city}
        
        # Her günü tek tek sözlüğe ekle
        for date_idx, val in daily_max.items():
            # Sütun ismi YYYY-MM-DD olacak
            col_name = date_idx.strftime("%Y-%m-%d")
            uv_row[col_name] = round(val, 2)
            
        uv_data_list.append(uv_row)

print("\n\n💾 Veriler işleniyor ve Excel'e kaydediliyor...")

if uv_data_list:
    df_final = pd.DataFrame(uv_data_list)
    
    # Sütun sayısını kontrol et (1110 gün civarı olması lazım)
    print(f"Toplam Gün Sayısı (Sütun): {len(df_final.columns) - 1}") 
    
    df_final.to_excel(FILE_NASA_DAILY, index=False)
    print(f"✅ NASA Günlük Max UV Raporu Hazır: {FILE_NASA_DAILY}")
else:
    print("❌ Kaydedilecek veri bulunamadı.")

print("\n🏁 Tüm işlemler tamamlandı.")

⏳ NASA Saatlik Veri -> Günlük MAX İşlemi Başlıyor... (20200101 - 20230115)
ℹ️  Amaç: Her günün en yüksek UV değerini (öğle vakti) yakalamak.
📂 Veriler 'sehir_verileri_nasa_hourly_37ay' klasörüne kaydedilecek.

📡 NASA Saatlik Veri İndiriliyor (Montevideo - Uruguay) - Bu işlem sürebilir...sürebilir......ebilir...

💾 Veriler işleniyor ve Excel'e kaydediliyor...
Toplam Gün Sayısı (Sütun): 1111
✅ NASA Günlük Max UV Raporu Hazır: Rapor_NASA_UV_Index_Gunluk_Max_Tam_Liste.xlsx

🏁 Tüm işlemler tamamlandı.
